In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import struct
import warnings

from typing import Tuple, Literal, Optional, List, Dict
# Importation des modules nécessaires de scikit-learn
from scipy.stats import entropy
from scipy.spatial.distance import euclidean
from scipy.spatial.distance import cityblock


from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score, average_precision_score
from sklearn.metrics import precision_recall_curve, auc, average_precision_score

from sklearn.base import clone
from sklearn.base import ClassifierMixin

# Classificateurs de scikit-learn
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    ElasticNetCV
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier,NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE


# LightGBM
import lightgbm

# Configuration du logging
import logging
# Ignorer les warnings
warnings.filterwarnings("ignore")


In [3]:
# Supprimer tous les handlers existants
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configurer le logger avec un format sans date/heure/niveau
logging.basicConfig(format='%(message)s', level=logging.INFO)

In [4]:
def calculate_pr_auc(y_true, y_prob):
    """
    Calcule et retourne le PR AUC pour une série de valeurs réelles et prédites.
    
    Args:
        y_true (array-like): Les valeurs réelles des classes (0 ou 1).
        y_prob (array-like): Les probabilités prédites des classes positives (entre 0 et 1).
    
    Returns:
        float: La valeur du PR AUC.
    """
    # Vérifier que y_true et y_prob sont compatibles
    if len(y_true) != len(y_prob):
        raise ValueError("La taille de y_true et y_prob doit être la même.")
    
    # Calcul de la courbe Precision-Recall
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)

    return pr_auc

In [5]:
def split_dataset(X, y, labeled_ratio, test_ratio):
    """
    Divise un dataset en train, test et pool en garantissant que chaque classe soit présente dans le set d'entraînement.
    """
    # Séparer l’ensemble de test
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=42, stratify=y
    )

    # Taille totale de l’ensemble train + pool
    train_pool_size = X_train_pool.shape[0]

    # Nombre d'échantillons à labelliser
    nb_labeled = int(labeled_ratio * train_pool_size)

    # Diviser train_pool en labeled et pool
    X_train, X_pool, y_train, y_pool = train_test_split(
        X_train_pool, y_train_pool, train_size=nb_labeled, random_state=42, stratify=y_train_pool
    )

    return X_train, X_pool, y_train, y_pool, X_test, y_test

In [6]:
def split_dataset_biased(X, y, labeled_ratio, test_ratio):
    """
    Divise un dataset en train, test et pool en garantissant que :
    - Le set de test est d'abord séparé.
    - Le train initial ne contient que les échantillons où X[:, 106] == 1.
    - On respecte les proportions demandées.
    """

    # Étape 1 : Séparation du test dès le début
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=42, stratify=y
    )

    # Taille totale après suppression du test
    train_pool_size = X_train_pool.shape[0]
    
    # Nombre d'échantillons à mettre dans train (en % de train+pool)
    nb_labeled = int(labeled_ratio * train_pool_size)

    # Étape 2 : Sélection des échantillons biaisés pour le train
    mask = X_train_pool[:, 106] == 0  # On filtre sur la feature 106
    X_biased = X_train_pool[mask]
    y_biased = y_train_pool[mask]

    #print(f"X_biased[106]{np.unique(X_biased[:,106],return_counts=True)}")
    #print(f"y_biased{np.unique(y_biased,return_counts=True)}")
    #print(f"shape de y_biased {y_biased.shape}")

    # Vérifier si assez d'exemples biaisés existent
    nb_labeled = min(nb_labeled, X_biased.shape[0])  # Ne pas dépasser ce qui est disponible
    print(f"nb_labeled {nb_labeled}")
    # Prendre les nb_labeled premiers éléments de la sélection biaisée
    X_train, y_train = X_biased[:nb_labeled], y_biased[:nb_labeled]

    # Étape 3 : Constituer le pool avec le reste des données non sélectionnées
    mask_remaining = np.ones(train_pool_size, dtype=bool)  # Tout est True au départ
    mask_remaining[np.where(mask)[0][:nb_labeled]] = False  # On enlève les éléments du train

    X_pool, y_pool = X_train_pool[mask_remaining], y_train_pool[mask_remaining]
    print(np.unique(y_train,return_counts=True))
    return X_train, X_pool, y_train, y_pool, X_test, y_test


In [7]:
def calculate_uncertainty(probabilities: np.ndarray, method: Literal["entropy", "margin", "least_confident", "random"] = "entropy") -> np.ndarray:
    """
    Calcule l'incertitude des prédictions basées sur les probabilités de classe.
    
    Paramètres :
        probabilities (np.ndarray) : Matrice des probabilités de classe pour chaque échantillon, de forme (n_samples, n_classes).
        method (str, optionnel) : Méthode d'incertitude à utiliser. Les options sont :
            - "entropy" : Entropie de Shannon.
            - "margin" : Différence entre les deux plus grandes probabilités.
            - "least_confident" : Complément de la plus grande probabilité.
            - "random" : Valeur aléatoire entre 0 et 1 pour chaque échantillon.
        
    Retourne :
        np.ndarray : Un tableau d'incertitude de forme (n_samples,) avec les scores d'incertitude pour chaque échantillon.
        
    Lève :
        ValueError : Si la méthode d'incertitude spécifiée n'est pas reconnue.
    """
    
    # Calcul de l'incertitude en fonction de la méthode choisie
    if method == "entropy":
        # Entropie de Shannon
        return entropy(probabilities.T, base=2)  # Probabilités doivent être de forme (n_classes, n_samples)
    
    elif method == "margin":
    # Si vous avez seulement deux classes
        if probabilities.shape[1] == 1:
            # Cas où la probabilité de chaque classe est un seul vecteur (probabilité = 1 pour une classe)
            return np.zeros(probabilities.shape[0])  # Pas de marge, car on est complètement confiant (différence = 0)
    
        elif probabilities.shape[1] == 2:
            # Cas classique avec deux classes
            prob_class_1 = probabilities[:, 0]
            prob_class_2 = probabilities[:, 1]
            return np.abs(prob_class_1 - prob_class_2)
    
        else:
            # Cas avec plus de deux classes, utiliser la marge entre les deux plus grandes probabilités
            sorted_probs = np.sort(probabilities, axis=1)
            return sorted_probs[:, -1] - sorted_probs[:, -2]

    
    elif method == "least_confident":
        # Complément de la probabilité la plus élevée (1 - max(probabilities))
        return 1 - np.max(probabilities, axis=1)
    
    elif method == "random":
        # Valeurs aléatoires entre 0 et 1
        return np.random.rand(probabilities.shape[0])
    
    else:
        # Si une méthode d'incertitude non reconnue est fournie, lever une exception
        raise ValueError("Méthode d'incertitude non reconnue")

In [8]:
def train_and_evaluate(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, model_class: type[ClassifierMixin],metric) -> tuple:
    """
    Entraîne un modèle sur un ensemble d'entraînement et évalue sa performance sur un ensemble de test.
    
    Paramètres :
        X_train (np.ndarray) : Les caractéristiques d'entraînement, de forme (n_samples, n_features).
        y_train (np.ndarray) : Les étiquettes d'entraînement, de forme (n_samples,).
        X_test (np.ndarray) : Les caractéristiques de test, de forme (n_samples, n_features).
        y_test (np.ndarray) : Les étiquettes de test, de forme (n_samples,).
        model_class (type) : La classe du modèle de classification (doit être un sous-type de `ClassifierMixin`, par exemple `RandomForestClassifier`).

    Retourne :
        tuple : Un tuple contenant :
            - Le modèle entraîné (`model`).
            - La précision sur l'ensemble de test (`accuracy`).
    """
    # Initialisation du modèle
    model = model_class()
    
    # Entraînement du modèle
    model.fit(X_train, y_train)
    
    # Prédiction sur l'ensemble de test
    y_pred = model.predict(X_test)
    
    # Si la métrique est F1, calculer F1-score
    if metric == f1_score:
        result = metric(y_test, y_pred, average="macro")
    
    # Si la métrique est "PR AUC", calculer PR AUC
    elif metric == "PR AUC":
        y_pred_prob = model.predict_proba(X_test)[:, 1]  # Probabilités pour la classe positive
        result = calculate_pr_auc(y_test, y_pred_prob)
    
    # Autres métriques
    else:
        result = metric(y_test, y_pred)
    
    return model, result

In [9]:
def select_uncertain_samples(model: ClassifierMixin, X_pool: np.ndarray, method: str, batch_size: int) -> np.ndarray:
    """
    Sélectionne les échantillons les plus incertains dans un ensemble de données non labellisées, en fonction de la méthode d'incertitude choisie.
    
    Paramètres :
        model (ClassifierMixin) : Le modèle d'apprentissage supervisé entraîné, avec une méthode `predict_proba` (par exemple, un classifieur comme `RandomForestClassifier` ou `SVC`).
        X_pool (np.ndarray) : Le tableau des caractéristiques des échantillons non labellisés, de forme (n_samples, n_features).
        method (str) : La méthode d'incertitude à utiliser pour calculer l'incertitude. Les options disponibles sont :
            - "entropy"
            - "margin"
            - "least_confident"
            - "random"
        batch_size (int) : Le nombre d'échantillons à sélectionner en fonction de l'incertitude.

    Retourne :
        np.ndarray : Un tableau des indices des `batch_size` échantillons les plus incertains dans `X_pool`.
    """
    # Probabilités prédites pour chaque échantillon
    probabilities = model.predict_proba(X_pool)
    
    # Calcul des incertitudes en fonction de la méthode choisie
    uncertainties = calculate_uncertainty(probabilities, method=method)
    
    # Sélection des indices des échantillons les plus incertains
    if method == "margin":
        # La méthode "margin" prend les échantillons ayant les plus petites marges entre les deux classes les plus probables
        return np.argsort(uncertainties)[:batch_size]
    else:
        # Pour les autres méthodes, on prend les échantillons ayant les incertitudes les plus élevées
        return np.argsort(uncertainties)[-batch_size:]

In [10]:
def update_labeled_unlabeled_sets(X_train: np.ndarray, y_train: np.ndarray, X_pool: np.ndarray, y_pool: np.ndarray, uncertain_indices: np.ndarray) -> tuple:
    """
    Met à jour les ensembles d'échantillons labellisés et non-labellisés après sélection des échantillons incertains.
    
    Paramètres :
        X_train (np.ndarray) : Ensemble des caractéristiques des échantillons labellisés (de taille (n_labeled_samples, n_features)).
        y_train (np.ndarray) : Ensemble des labels des échantillons labellisés (de taille (n_labeled_samples,)).
        X_pool (np.ndarray) : Ensemble des caractéristiques des échantillons non labellisés (de taille (n_pool_samples, n_features)).
        y_pool (np.ndarray) : Ensemble des labels des échantillons non labellisés (de taille (n_pool_samples,)).
        uncertain_indices (np.ndarray) : Indices des échantillons les plus incertains dans `X_pool`, qui doivent être ajoutés à `X_train` (de taille (batch_size,)).

    Retourne :
        tuple : 
            - X_train (np.ndarray) : Ensemble mis à jour des caractéristiques des échantillons labellisés.
            - y_train (np.ndarray) : Ensemble mis à jour des labels des échantillons labellisés.
            - X_pool (np.ndarray) : Ensemble mis à jour des caractéristiques des échantillons non labellisés.
            - y_pool (np.ndarray) : Ensemble mis à jour des labels des échantillons non labellisés.
    """
    # Ajout des échantillons incertains à l'ensemble des échantillons labellisés
    X_train = np.vstack((X_train, X_pool[uncertain_indices]))  # Empile les échantillons incertains sur l'ensemble de train
    y_train = np.hstack((y_train, y_pool[uncertain_indices]))  # Ajoute les labels des échantillons incertains

    # Création d'un masque pour supprimer les échantillons incertains de l'ensemble non-labellisé
    mask = np.ones(len(X_pool), dtype=bool)  # Crée un masque de True pour tous les échantillons non labellisés
    mask[uncertain_indices] = False  # Marque les indices incertains comme False dans le masque
    
    # Mise à jour de l'ensemble non-labellisé en excluant les échantillons incertains
    X_pool = X_pool[mask]  # Ne garde que les échantillons non incertains
    y_pool = y_pool[mask]  # Ne garde que les labels des échantillons non incertains

    return X_train, y_train, X_pool, y_pool

In [11]:
def hybrid_uncertainty(model, X_pool: np.ndarray, w1: float = 0.33, w2: float = 0.33, w3: float = 0.34) -> np.ndarray:
    """
    Combine plusieurs mesures d'incertitude (confiance minimale, marge et entropie) en une seule mesure pondérée.
    
    Args:
        model (sklearn.base.BaseEstimator): Le modèle de classification entraîné.
        X_pool (np.ndarray): Les échantillons non labellisés (de taille (n_pool_samples, n_features)).
        w1 (float, optional): Poids de la mesure "least confident" (par défaut 0.33).
        w2 (float, optional): Poids de la mesure "margin" (par défaut 0.33).
        w3 (float, optional): Poids de la mesure "entropy" (par défaut 0.34).
    
    Returns:
        np.ndarray: Un tableau des scores d'incertitude combinés (de taille (n_pool_samples,)).
    """
    # Prédictions des probabilités pour chaque classe
    proba = model.predict_proba(X_pool)
    
    # Mesure d'incertitude "Least Confident" : 1 - probabilité maximale
    least_confident = 1 - np.max(proba, axis=1)
    
    # Mesure "Margin" : différence entre les deux plus grandes probabilités
    if proba.shape[1] == 1:
        # Si on a qu'une seule classe (par exemple, probabilité = 1 pour une seule classe), on ne peut pas calculer une marge
        margin = np.zeros(proba.shape[0])  # Pas de marge dans ce cas
    else:
        sorted_proba = np.sort(proba, axis=1)
        margin = sorted_proba[:, -1] - sorted_proba[:, -2]
    
    # Entropie des probabilités : mesure de l'incertitude globale
    epsilon = 1e-10  # Ajout d'un petit epsilon pour éviter log(0)
    entropy = -np.sum(proba * np.log(proba + epsilon), axis=1)
    
    # Combinaison des scores d'incertitude avec les poids
    combined_score = w1 * least_confident + w2 * (1 - margin) + w3 * entropy
    
    return combined_score


In [12]:
def select_uncertain_samples_hybrid(model, X_pool: np.ndarray, batch_size: int) -> np.ndarray:
    """
    Sélectionne les échantillons les plus incertains selon la stratégie hybride en combinant 
    plusieurs mesures d'incertitude.
    
    Args:
        model (sklearn.base.BaseEstimator): Le modèle de classification entraîné.
        X_pool (np.ndarray): Les échantillons non labellisés (de taille (n_pool_samples, n_features)).
        batch_size (int): Nombre d'échantillons à sélectionner pour chaque itération.
    
    Returns:
        np.ndarray: Indices des échantillons les plus incertains dans X_pool (de taille (batch_size,)).
    """
    # Calcul des scores d'incertitude combinés (stratégie hybride)
    scores = hybrid_uncertainty(model, X_pool)
    
    # Sélectionner les indices des échantillons les plus incertains
    return np.argsort(scores)[-batch_size:]  # Retourne les indices des scores d'incertitude les plus élevés

In [13]:
def select_uncertain_samples_qbc(method: str, models: list, X_train: np.ndarray, y_train: np.ndarray, 
                                 X_pool: np.ndarray, batch_size: int) -> np.ndarray:
    """
    Sélectionne les échantillons les plus incertains en utilisant le Query by Committee (QBC).
    
    Le QBC évalue l'incertitude des échantillons dans l'ensemble non-labellisé en comparant les 
    prédictions des modèles d'un comité. Trois méthodes sont supportées :
    - "qbc-variance" : variance des prédictions des modèles sur chaque échantillon.
    - "qbc-entropy" : entropie des votes du comité.
    - "qbc-KL" : divergence Kullback-Leibler entre les prédictions des modèles et la probabilité moyenne.
    
    Args:
        method (str): Méthode d'incertitude à utiliser. Une des options suivantes :
                      - "qbc-variance"
                      - "qbc-entropy"
                      - "qbc-KL"
        models (list): Liste des modèles (du comité), chacun devant avoir la méthode `predict_proba`.
        X_train (np.ndarray): Ensemble de données d'entraînement labellisées (n_samples, n_features).
        y_train (np.ndarray): Labels des données d'entraînement (n_samples,).
        X_pool (np.ndarray): Données non-labellisées (n_pool_samples, n_features).
        batch_size (int): Nombre d'échantillons à sélectionner dans chaque itération.
    
    Returns:
        np.ndarray: Indices des échantillons les plus incertains dans `X_pool` (taille = batch_size).
    """
    # Entraîner chaque modèle du comité sur les données labellisées
    for model in models:
        model.fit(X_train, y_train)

    # Obtenir les prédictions de chaque modèle sur le pool de données non labellisées
    predictions = np.array([model.predict_proba(X_pool) for model in models])  # Shape: (n_models, n_samples, n_classes)

    # Initialiser l'incertitude
    uncertainty = None

    if method == "qbc-variance":
        # Calcul de la variance des prédictions pour chaque échantillon
        uncertainty = np.var(predictions, axis=0)  # Variance des probabilités
        uncertainty = np.mean(uncertainty, axis=1)  # Moyenne de la variance sur les classes

    elif method == "qbc-entropy":
        # Comptage des votes pour chaque classe
        n_models, n_samples, n_classes = predictions.shape
        vote_counts = np.zeros((n_samples, n_classes))

        for i in range(n_samples):
            for j in range(n_models):
                # Trouver la classe prédite par chaque modèle
                predicted_class = np.argmax(predictions[j, i, :])
                vote_counts[i, predicted_class] += 1

        # Calcul des probabilités de vote
        vote_probs = vote_counts / n_models

        # Calcul de l'entropie des votes pour chaque échantillon
        uncertainty = np.array([entropy(vote_probs[i, :], base=2) for i in range(n_samples)])

    elif method == "qbc-KL":
        # Calcul de la probabilité moyenne (P_avg) pour chaque échantillon
        P_avg = np.mean(predictions, axis=0)  # Moyenne des prédictions des modèles (n_samples, n_classes)

        # Calcul de la divergence Kullback-Leibler (KL) pour chaque échantillon
        n_samples, n_classes = P_avg.shape
        uncertainty = np.zeros(n_samples)

        for i in range(n_samples):
            for model_preds in predictions[:, i, :]:
                # Calcul de la divergence KL pour chaque échantillon
                uncertainty[i] += np.sum(model_preds * np.log(model_preds / P_avg[i, :]))

    else:
        raise ValueError("Méthode d'incertitude non reconnue. Choisissez parmi 'qbc-variance', 'qbc-entropy' ou 'qbc-KL'.")

    # Sélectionner les indices des échantillons ayant l'incertitude la plus élevée
    uncertain_indices = np.argsort(uncertainty)[-batch_size:]

    return uncertain_indices

In [14]:
#à terminer
def select_uncertain_samples_egl(model, X_pool: np.ndarray, batch_size: int) -> np.ndarray:
    """
    Sélectionne les échantillons les plus informatifs en utilisant la méthode EGL (Expected Gradient Length).
    
    Cette méthode sélectionne les échantillons qui entraîneraient la plus grande mise à jour du modèle
    en fonction de la norme du gradient de la fonction de coût par rapport à chaque échantillon.
    
    Args:
        model : Estimation du modèle de classification (doit être basé sur des gradients, ex : régression logistique).
        X_pool : Données non-labellisées pour lesquelles nous souhaitons calculer l'incertitude.
        batch_size : Nombre d'échantillons à sélectionner.
    
    Returns:
        np.ndarray : Indices des échantillons les plus incertains dans `X_pool`.
    """
    if not hasattr(model, "coef_"):
        raise ValueError("Le modèle doit être une régression logistique ou un autre modèle basé sur des gradients.")
    
    gradients = []
    
    # Obtenir les probabilités de prédiction
    probs = model.predict_proba(X_pool)  # Probabilités pour chaque classe (n_samples, n_classes)
    
    # Calculer l'incertitude pour chaque échantillon en utilisant la méthode EGL
    for i, x in enumerate(X_pool):
        expected_grad = 0
        
        for class_idx in range(probs.shape[1]):  # Parcourir chaque classe
            prob = probs[i, class_idx]
            
            # Créer un vecteur one-hot pour la classe cible
            y_dummy = np.zeros((1, probs.shape[1]))
            y_dummy[0, class_idx] = 1
            
            try:
                # Calcul du gradient pour cette classe en ajustant temporairement le modèle
                model.fit(x.reshape(1, -1), y_dummy.argmax(axis=1))  # Entraînement temporaire sur l'exemple (1, -1) pour l'ajustement de la forme
                grad = np.linalg.norm(model.coef_)  # Norme du gradient
                expected_grad += prob * grad  # Calcul de l'espérance du gradient
            except ValueError:
                # Si une erreur survient, on passe à la classe suivante
                continue
        
        gradients.append(expected_grad)
    
    # Sélection des indices des plus grandes valeurs de gradient
    uncertain_indices = np.argsort(gradients)[-batch_size:]
    
    return uncertain_indices


In [15]:
def compute_information_density(X_pool: np.ndarray, similarity_metric: str = 'cosine') -> np.ndarray:
    """
    Calcule la densité d'information pour chaque point du pool en fonction de la similarité avec les autres points.
    
    Args:
        X_pool (numpy.ndarray): Ensemble non labellisé de taille (n_samples, n_features).
        similarity_metric (str): Type de mesure de similarité ('cosine' ou 'euclidean').
    
    Returns:
        numpy.ndarray: Score de densité pour chaque instance.
    """
    # Calcul de la matrice de similarité en fonction de la métrique choisie
    if similarity_metric == 'cosine':
        similarity_matrix = cosine_similarity(X_pool)
    elif similarity_metric == 'euclidean':
        distance_matrix = np.linalg.norm(X_pool[:, np.newaxis] - X_pool, axis=2)
        similarity_matrix = 1 / (1 + distance_matrix)  # Conversion en similarité
    else:
        raise ValueError("Metric non supportée. Utilisez 'cosine' ou 'euclidean'.")
    
    # Calcul de la moyenne des similarités pour chaque point (densité)
    # Evite l'auto-similarité (similarité d'un point avec lui-même) en mettant la diagonale à zéro
    np.fill_diagonal(similarity_matrix, 0)
    
    density_scores = np.mean(similarity_matrix, axis=1)
    
    # Normalisation des scores de densité entre 0 et 1 (facultatif)
    density_scores /= np.max(density_scores)
    
    return density_scores

In [16]:
def select_uncertain_samples_density(model, X_pool: np.ndarray, batch_size: int, similarity_metric: str = 'cosine') -> list:
    """
    Sélectionne les échantillons en combinant incertitude et densité d'information.
    
    Args:
        model : Modèle de classification entraîné utilisé pour l'incertitude.
        X_pool : Ensemble non labellisé de taille (n_samples, n_features).
        batch_size : Nombre d'échantillons à sélectionner.
        similarity_metric : Type de mesure de similarité ('cosine' ou 'euclidean').
    
    Returns:
        list: Indices des échantillons sélectionnés.
    """
    # 1. Vérification de la méthode de calcul des probabilités pour l'incertitude
    if not hasattr(model, "predict_proba"):
        raise ValueError("Le modèle doit implémenter 'predict_proba' pour calculer l'incertitude.")
    
    probas = model.predict_proba(X_pool)
    # Calcul de l'incertitude (exemple : entropie prédictive)
    uncertainty = -np.sum(probas * np.log(probas + 1e-10), axis=1)  # Évite log(0)
    
    # 2. Calcul de la densité d'information
    density_scores = compute_information_density(X_pool, similarity_metric)
    
    # 3. Combinaison des scores : pondération entre incertitude et densité
    combined_scores = uncertainty * density_scores
    
    # 4. Sélection des indices des échantillons les plus incertains et informatifs
    selected_indices = np.argsort(combined_scores)[-batch_size:]
    
    return selected_indices.tolist()

In [17]:
def select_uncertain_samples_by_distance_density(method, model, X_pool, batch_size, models, X_train, y_train, similarity_metric='cityblock'):
    """
    Sélectionne les échantillons les plus éloignés des autres clusters, tout en tenant compte de la densité.
    
    Parameters:
    - method : méthode d'Active Learning (utile pour l'interface mais non utilisée dans cette méthode).
    - model : modèle de classification utilisé pour l'Active Learning.
    - X_pool : ensemble des données non labellisées.
    - batch_size : nombre d'échantillons à ajouter au training set.
    - models : liste de modèles pour les méthodes basées sur un comité (utile dans certains cas de AL).
    - X_train : ensemble d'entraînement labellisé.
    - y_train : labels de l'ensemble d'entraînement.
    - similarity_metric : critère de similarité pour calculer les distances (par défaut 'cosine').
    
    Returns:
    - indices des échantillons sélectionnés dans X_pool.
    """
    # Étape 1: Calculer les distances des points dans le pool par rapport aux points labellisés.
    nn = NearestNeighbors(n_neighbors=1, metric=similarity_metric)
    nn.fit(X_train)  # Apprendre les distances des points labellisés
    
    # Trouver la distance minimale pour chaque point dans le pool par rapport à l'ensemble labellisé
    distances, indices = nn.kneighbors(X_pool)
    
    # Étape 2: Calculer la densité des points du pool en utilisant le k-nearest neighbors (KNN)
    density_nn = NearestNeighbors(n_neighbors=5, metric=similarity_metric)  # Utilisation de 5 voisins pour estimer la densité
    density_nn.fit(X_pool)
    densities, _ = density_nn.kneighbors(X_pool)  # Obtient la distance des 5 voisins les plus proches
    densities = densities.mean(axis=1)  # Moyenne des distances des voisins, plus élevé = plus densément peuplé
    
    # Étape 3: Calculer une métrique combinée basée sur la distance et la densité
    # Les points éloignés des autres clusters et ayant une faible densité sont considérés comme plus incertains
    score = distances.flatten() / (densities + 1e-5)  # Pour éviter la division par zéro

    # Étape 4: Sélectionner les indices des points avec les scores les plus élevés (points éloignés et peu denses)
    uncertain_indices = np.argsort(score)[-batch_size:]  # Sélectionner les "batch_size" plus incertains
    
    return uncertain_indices

In [18]:
def select_uncertain_samples_general(method: str, model: object, X_pool: np.ndarray, batch_size: int, models: Optional[List[object]] = None, X_train: Optional[np.ndarray] = None, y_train: Optional[np.ndarray] = None, similarity_metric: str = 'cosine') -> np.ndarray:
    """
    Sélectionne les échantillons les plus incertains selon la méthode spécifiée.
    
    Args:
        method (str): Méthode de sélection. Doit être l'une des suivantes : 
            'qbc-variance', 'qbc-entropy', 'qbc-KL', 'hybrid', 'EGL', 'least_confident', 
            'margin', 'entropy', 'random', 'density'.
        model (object): Modèle de classification entraîné (ex. RandomForestClassifier, LogisticRegression, etc.).
        X_pool (numpy.ndarray): Ensemble non labellisé de taille (n_samples, n_features).
        batch_size (int): Nombre d'échantillons à sélectionner.
        models (list, optionnel): Liste de modèles pour la méthode QBC. Requis pour 'qbc-*'.
        X_train (numpy.ndarray, optionnel): Données d'entraînement labellisées. Requis pour QBC.
        y_train (numpy.ndarray, optionnel): Labels d'entraînement. Requis pour QBC.
        similarity_metric (str, optionnel): Métrique de similarité pour 'density'. Valeurs possibles : 'cosine', 'euclidean'.
    
    Returns:
        numpy.ndarray: Indices des échantillons les plus incertains sélectionnés.
    
    Raises:
        ValueError: Si la méthode est inconnue ou si les arguments nécessaires sont manquants.
    """
    
    # Vérification des arguments nécessaires pour certaines méthodes
    if method in ["qbc-variance", "qbc-entropy", "qbc-KL"]:
        if models is None or X_train is None or y_train is None:
            raise ValueError("Les méthodes QBC nécessitent un comité de modèles et des données d'entraînement labellisées.")
        return select_uncertain_samples_qbc(method, models, X_train, y_train, X_pool, batch_size)
    
    elif method == "hybrid":
        return select_uncertain_samples_hybrid(model, X_pool, batch_size)
    
    elif method == "EGL":
        return select_uncertain_samples_egl(model, X_pool, batch_size)
    
    elif method in ["least_confident", "margin", "entropy"]:
        return select_uncertain_samples(model, X_pool, method, batch_size)
    
    elif method == "random":
        # Sélection aléatoire d'échantillons sans remplacement
        return np.random.choice(len(X_pool), size=batch_size, replace=False)
    
    elif method == "density":
        return select_uncertain_samples_density(model, X_pool, batch_size, similarity_metric)
    
    elif method =="distance_density":
        return(select_uncertain_samples_by_distance_density(method, model, X_pool, batch_size, models, X_train, y_train, similarity_metric='cosine'))
    
    else:
        raise ValueError(f"Méthode de sélection non reconnue : {method}")


In [19]:
def plot_active_learning_results(dataset_name: str, methods: List[str], accuracies: Dict[str, Dict[str, List[float]]],
                                 labeled_ratio: float, batch_ratio: float, n_iterations: int, 
                                 full_training_accuracies: Dict[str, float], final_accuracies: Dict[str, Dict[str, float]], 
                                 initial_accuracies: Dict[str, float],metric) -> Dict[str, float]:
    """
    Génère et affiche les graphiques d'évolution de l'accuracy et d'amélioration de l'accuracy pour un dataset donné.
    
    Args:
        dataset_name (str): Nom du dataset.
        methods (list): Liste des méthodes utilisées.
        accuracies (dict): Dictionnaire des accuracies obtenues par méthode pour chaque dataset.
        labeled_ratio (float): Ratio initial de données labellisées.
        batch_ratio (float): Ratio de batch d'ajout de labels à chaque itération.
        n_iterations (int): Nombre total d'itérations.
        full_training_accuracies (dict): Accuracy du modèle entraîné sur tout le training set pour chaque dataset.
        final_accuracies (dict): Dictionnaire des accuracies finales obtenues par chaque méthode après toutes les itérations.
        initial_accuracies (dict): Dictionnaire des accuracies initiales avant le processus d'Active Learning.
    
    Returns:
        dict: Dictionnaire des améliorations d'accuracy pour chaque méthode.
    """
    
    # === Courbe d'évolution de l'accuracy ===
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")
    
    
    # Tracer l'évolution de l'accuracy pour chaque méthode
    for method in methods:
        # Ajout de l'accuracy initiale avant la première itération
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(n_iterations+1)]
        
        y_values = accuracies[dataset_name][method]
        
        fig.add_scatter(x=x_values, y=y_values, mode='lines+markers', name=method)
    
    # Ajouter la performance du modèle entraîné sur tout le training set
    fig.add_scatter(
        x=[labeled_ratio * 100, (labeled_ratio + (n_iterations+1) * batch_ratio) * 100],
        y=[full_training_accuracies[dataset_name]] * 2,
        mode='lines', name="Full Training Set", line=dict(dash='dash')
    )
    
    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[labeled_ratio * 100, (labeled_ratio + n_iterations * batch_ratio) * 100]),
        yaxis=dict(range=[min(min(accuracies[dataset_name][m]) for m in methods) - 0.02, 
                          max(full_training_accuracies[dataset_name], 
                              max(final_accuracies[dataset_name].values())) + 0.02])
    )
    fig.show()
    
    # === Barres d'amélioration de l'accuracy ===
    # Calculer l'amélioration de l'accuracy pour chaque méthode
    accuracy_improvements = {
        method: final_accuracies[dataset_name][method] - initial_accuracies[dataset_name] 
        for method in methods
    }
    
    # Tracer un graphique en barres montrant l'amélioration de l'accuracy
    fig = px.bar(
        x=methods, 
        y=[accuracy_improvements[m] for m in methods],
        labels={"x": "Méthode", "y": f"Amélioration de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)}"},
        title=f"Amélioration de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}"
    )

    
    # Ajuster les limites de l'axe des y pour un meilleur affichage
    fig.update_layout(
        yaxis=dict(range=[min(accuracy_improvements.values()) - 0.01, 
                          max(accuracy_improvements.values()) + 0.01])
    )
    fig.show()
    
    return accuracy_improvements


In [20]:
def run_active_learning_experiment_datasets(datasets: dict, METRICS: dict,labeled_ratio: float, test_ratio: float, 
                                            n_iterations: int, batch_ratio: float, methods: list, 
                                            model_class: type, models: list) -> dict:
    """
    Exécute une expérimentation d'Active Learning sur plusieurs datasets avec différentes méthodes de sélection d'incertitude.
    
    Paramètres:
        datasets (dict): Un dictionnaire contenant les datasets sous la forme {nom_dataset: (X, y)},
                         où X est un np.ndarray (features) et y est un np.ndarray (labels).
        labeled_ratio (float): Proportion initiale d'échantillons labellisés dans l'ensemble d'entraînement.
        test_ratio (float): Proportion d'échantillons dédiés à l'ensemble de test.
        n_iterations (int): Nombre d'itérations d'Active Learning.
        batch_ratio (float): Proportion d'échantillons ajoutés à chaque itération (par rapport à la taille totale du training dataset).
        methods (list): Liste des méthodes de sélection d'incertitude à tester.
        model_class (type): Classe du modèle de classification utilisé (doit être compatible avec `fit` et `predict`).
        models (list): Liste des modèles pour les méthodes basées sur un comité (ex: Query By Committee).

    Retourne:
        dict: Un dictionnaire contenant les précisions finales pour chaque dataset et chaque méthode d'Active Learning.
    """
    
    # Initialisation des dictionnaires pour stocker les résultats
    results= {dataset_name: {method: [] for method in methods} for dataset_name in datasets.keys()}
    final_results = {dataset_name: {} for dataset_name in datasets.keys()}
    initial_results = {}
    full_training_results = {}
    
    for dataset_name, (X, y) in datasets.items():
        metric=METRICS[dataset_name]
        logging.info(f"\nTraitement du dataset: {dataset_name}")
        logging.info(f"\nMétrique utilisée: {metric.__name__ if hasattr(metric, '__name__') else str(metric)}")
        # Création des ensembles d'entraînement et de test
        X_train, X_pool, y_train, y_pool, X_test, y_test = split_dataset(X, y, labeled_ratio, test_ratio)
        batch_size = int(batch_ratio * (X_train.shape[0]+X_pool.shape[0]))
        
        total_samples = X.shape[0]
        labeled_percentage = (X_train.shape[0] /(X_train.shape[0]+X_pool.shape[0])) * 100
        unlabeled_percentage = (X_pool.shape[0] / (X_train.shape[0]+X_pool.shape[0])) * 100
        test_percentage = (X_test.shape[0] / total_samples) * 100

        # Affichage des tailles des ensembles
        total_samples = X.shape[0]
        logging.info(f"Taille totale du dataset {dataset_name}: {total_samples/total_samples*100:.2f}% ({total_samples}/{total_samples})")
        logging.info(f"Taille de l'ensemble de test: {test_percentage:.2f}% ({X_test.shape[0]}/{total_samples})")
        logging.info(f"Taille de l'ensemble de training: {100-test_percentage:.2f}% ({X_train.shape[0]+X_pool.shape[0]}/{total_samples})")     
        logging.info(f"Taille de l'ensemble labellisé dans le training set: {labeled_percentage:.2f}% ({X_train.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Taille de l'ensemble non-labellisé dans le training set: {unlabeled_percentage:.2f}% ({X_pool.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Nb d'itérations: {n_iterations}")
        logging.info(f"Nb de données labellisées en plus à chaque itération: {batch_size/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({batch_size}/{X_train.shape[0]+X_pool.shape[0]})")
        
        # Évaluation initiale
        model = model_class()
        y_pred_initial = model.fit(X_train, y_train).predict(X_test)
        if metric == "PR AUC":
            #print("Classes présentes dans y_train:", np.unique(y_train, return_counts=True))
            probas = model.predict_proba(X_test)
            #print("Shape de predict_proba:", probas.shape)

            y_pred_prob_initial = model.predict_proba(X_test)[:, 1]
            initial_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_initial)
        else:
            initial_results[dataset_name] = (
                metric(y_test, y_pred_initial, average="macro") if metric == f1_score else metric(y_test, y_pred_initial)
            )
        

        # Modèle entraîné sur l'ensemble complet des données d'entraînement
        model_full = model_class()
        model_full.fit(np.vstack((X_train, X_pool)), np.hstack((y_train, y_pool)))
        if metric == "PR AUC":
            y_pred_prob_full = model_full.predict_proba(X_test)[:, 1]
            full_training_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_full)
        else:
            full_training_results[dataset_name] = (
                metric(y_test, model_full.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model_full.predict(X_test))
            )

        # === Boucle principale d'Active Learning ===
        for method in methods:
            X_train_temp, y_train_temp = X_train.copy(), y_train.copy()
            X_pool_temp, y_pool_temp = X_pool.copy(), y_pool.copy()
            results[dataset_name][method].append(initial_results[dataset_name])
            for i in range(n_iterations):
                # Entraînement et évaluation du modèle
                model, acc = train_and_evaluate(X_train_temp, y_train_temp, X_test, y_test, model_class,metric)
                results[dataset_name][method].append(acc)
                
                # Vérification de l'existence d'échantillons non labellisés
                if len(X_pool_temp) == 0:
                    logging.info(f"\nToutes les données ont été labellisées après {i} itérations.")
                    break
                
                # Ajustement de la taille du batch si nécessaire
                actual_batch_size = min(batch_size, len(X_pool_temp))
                if actual_batch_size < batch_size:
                    logging.info(f"\nBatch réduit à {actual_batch_size} échantillons car la pool est presque vide.")
                
                # Sélection des échantillons les plus incertains
                uncertain_indices = select_uncertain_samples_general(method, model, X_pool_temp, actual_batch_size, models, X_train_temp, y_train_temp, similarity_metric='cosine')
                
                # Mise à jour des ensembles labellisés et non-labellisés
                X_train_temp, y_train_temp, X_pool_temp, y_pool_temp = update_labeled_unlabeled_sets(
                    X_train_temp, y_train_temp, X_pool_temp, y_pool_temp, uncertain_indices
                )
                
                #logging.info(f"{method} - Iteration {i+1}: {len(X_train_temp)/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({len(X_train_temp)}/{(X_train.shape[0]+X_pool.shape[0])}) samples labeled, {metric.__name__ if hasattr(metric, '__name__') else str(metric)}: {acc:.4f} on {dataset_name}")
            
            # Final evaluation after Active Learning
            if metric == "PR AUC":
                y_pred_prob_final = model.predict_proba(X_test)[:, 1]
                pr_auc = calculate_pr_auc(y_test, y_pred_prob_final)
                final_results[dataset_name][method] = pr_auc
                #logging.info(f"Final PR AUC ({method}) on {dataset_name}: {pr_auc:.4f}")
            else:
                final_results[dataset_name][method] = (
                    metric(y_test, model.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model.predict(X_test))
                )
                logging.info(f"Final {metric.__name__ if hasattr(metric, '__name__') else str(metric)} ({method}) on {dataset_name}: {final_results[dataset_name][method]:.4f}")        
        #Génération des graphiques et affichage des résultats
        result_improvements = plot_active_learning_results(dataset_name, methods, results, labeled_ratio, batch_ratio,i+1, full_training_results, final_results, initial_results,metric)
        
        #logging.info(f"Initial {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {initial_results[dataset_name]:.4f}")
        #logging.info(f"Full Training Set {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {full_training_results[dataset_name]:.4f}")

        #for method in methods:
           #logging.info(f"{metric.__name__ if hasattr(metric, '__name__') else str(metric)} Improvement ({method}) on {dataset_name}: {result_improvements[method]:.4f}")
    
    return results

In [21]:
def run_active_learning_experiment_datasets_biased(datasets: dict, METRICS: dict,labeled_ratio: float, test_ratio: float, 
                                            n_iterations: int, batch_ratio: float, methods: list, 
                                            model_class: type, models: list) -> dict:
    """
    Exécute une expérimentation d'Active Learning sur plusieurs datasets avec différentes méthodes de sélection d'incertitude.
    
    Paramètres:
        datasets (dict): Un dictionnaire contenant les datasets sous la forme {nom_dataset: (X, y)},
                         où X est un np.ndarray (features) et y est un np.ndarray (labels).
        labeled_ratio (float): Proportion initiale d'échantillons labellisés dans l'ensemble d'entraînement.
        test_ratio (float): Proportion d'échantillons dédiés à l'ensemble de test.
        n_iterations (int): Nombre d'itérations d'Active Learning.
        batch_ratio (float): Proportion d'échantillons ajoutés à chaque itération (par rapport à la taille totale du training dataset).
        methods (list): Liste des méthodes de sélection d'incertitude à tester.
        model_class (type): Classe du modèle de classification utilisé (doit être compatible avec `fit` et `predict`).
        models (list): Liste des modèles pour les méthodes basées sur un comité (ex: Query By Committee).

    Retourne:
        dict: Un dictionnaire contenant les précisions finales pour chaque dataset et chaque méthode d'Active Learning.
    """
    
    # Initialisation des dictionnaires pour stocker les résultats
    results= {dataset_name: {method: [] for method in methods} for dataset_name in datasets.keys()}
    final_results = {dataset_name: {} for dataset_name in datasets.keys()}
    initial_results = {}
    full_training_results = {}
    
    for dataset_name, (X, y) in datasets.items():
        metric=METRICS[dataset_name]
        logging.info(f"\nTraitement du dataset: {dataset_name}")
        logging.info(f"\nMétrique utilisée: {metric.__name__ if hasattr(metric, '__name__') else str(metric)}")
        # Création des ensembles d'entraînement et de test
        X_train, X_pool, y_train, y_pool, X_test, y_test = split_dataset_biased(X, y, labeled_ratio, test_ratio)
        batch_size = int(batch_ratio * (X_train.shape[0]+X_pool.shape[0]))
        
        total_samples = X.shape[0]
        labeled_percentage = (X_train.shape[0] /(X_train.shape[0]+X_pool.shape[0])) * 100
        unlabeled_percentage = (X_pool.shape[0] / (X_train.shape[0]+X_pool.shape[0])) * 100
        test_percentage = (X_test.shape[0] / total_samples) * 100

        # Affichage des tailles des ensembles
        total_samples = X.shape[0]
        logging.info(f"Taille totale du dataset {dataset_name}: {total_samples/total_samples*100:.2f}% ({total_samples}/{total_samples})")
        logging.info(f"Taille de l'ensemble de test: {test_percentage:.2f}% ({X_test.shape[0]}/{total_samples})")
        logging.info(f"Taille de l'ensemble de training: {100-test_percentage:.2f}% ({X_train.shape[0]+X_pool.shape[0]}/{total_samples})")     
        logging.info(f"Taille de l'ensemble labellisé dans le training set: {labeled_percentage:.2f}% ({X_train.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Taille de l'ensemble non-labellisé dans le training set: {unlabeled_percentage:.2f}% ({X_pool.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Nb d'itérations: {n_iterations}")
        logging.info(f"Nb de données labellisées en plus à chaque itération: {batch_size/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({batch_size}/{X_train.shape[0]+X_pool.shape[0]})")
        print("Classes présentes dans xtrain 106:", np.unique(X_train[:,106], return_counts=True))
        
        # Évaluation initiale
        model = model_class()
        y_pred_initial = model.fit(X_train, y_train).predict(X_test)
        if metric == "PR AUC":
            print("Classes présentes dans y_train:", np.unique(y_train, return_counts=True))
            probas = model.predict_proba(X_test)
            print("Shape de predict_proba:", probas.shape)

            y_pred_prob_initial = model.predict_proba(X_test)[:, 1]
            initial_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_initial)
        else:
            initial_results[dataset_name] = (
                metric(y_test, y_pred_initial, average="macro") if metric == f1_score else metric(y_test, y_pred_initial)
            )
        

        # Modèle entraîné sur l'ensemble complet des données d'entraînement
        model_full = model_class()
        model_full.fit(np.vstack((X_train, X_pool)), np.hstack((y_train, y_pool)))
        if metric == "PR AUC":
            y_pred_prob_full = model_full.predict_proba(X_test)[:, 1]
            full_training_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_full)
        else:
            full_training_results[dataset_name] = (
                metric(y_test, model_full.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model_full.predict(X_test))
            )

        # === Boucle principale d'Active Learning ===
        for method in methods:
            X_train_temp, y_train_temp = X_train.copy(), y_train.copy()
            X_pool_temp, y_pool_temp = X_pool.copy(), y_pool.copy()
            results[dataset_name][method].append(initial_results[dataset_name])
            for i in range(n_iterations):
                # Entraînement et évaluation du modèle
                model, acc = train_and_evaluate(X_train_temp, y_train_temp, X_test, y_test, model_class,metric)
                results[dataset_name][method].append(acc)
                
                # Vérification de l'existence d'échantillons non labellisés
                if len(X_pool_temp) == 0:
                    logging.info(f"\nToutes les données ont été labellisées après {i} itérations.")
                    break
                
                # Ajustement de la taille du batch si nécessaire
                actual_batch_size = min(batch_size, len(X_pool_temp))
                if actual_batch_size < batch_size:
                    logging.info(f"\nBatch réduit à {actual_batch_size} échantillons car la pool est presque vide.")
                
                # Sélection des échantillons les plus incertains
                uncertain_indices = select_uncertain_samples_general(method, model, X_pool_temp, actual_batch_size, models, X_train_temp, y_train_temp, similarity_metric='cosine')
                
                # Mise à jour des ensembles labellisés et non-labellisés
                X_train_temp, y_train_temp, X_pool_temp, y_pool_temp = update_labeled_unlabeled_sets(
                    X_train_temp, y_train_temp, X_pool_temp, y_pool_temp, uncertain_indices
                )
                
                logging.info(f"{method} - Iteration {i+1}: {len(X_train_temp)/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({len(X_train_temp)}/{(X_train.shape[0]+X_pool.shape[0])}) samples labeled, {metric.__name__ if hasattr(metric, '__name__') else str(metric)}: {acc:.4f} on {dataset_name}")
            
            # Final evaluation after Active Learning
            if metric == "PR AUC":
                y_pred_prob_final = model.predict_proba(X_test)[:, 1]
                pr_auc = calculate_pr_auc(y_test, y_pred_prob_final)
                final_results[dataset_name][method] = pr_auc
                logging.info(f"Final PR AUC ({method}) on {dataset_name}: {pr_auc:.4f}")
            else:
                final_results[dataset_name][method] = (
                    metric(y_test, model.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model.predict(X_test))
                )
                logging.info(f"Final {metric.__name__ if hasattr(metric, '__name__') else str(metric)} ({method}) on {dataset_name}: {final_results[dataset_name][method]:.4f}")        
        #Génération des graphiques et affichage des résultats
        #result_improvements = plot_active_learning_results(dataset_name, methods, results, labeled_ratio, batch_ratio,i+1, full_training_results, final_results, initial_results,metric)
        
        logging.info(f"Initial {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {initial_results[dataset_name]:.4f}")
        logging.info(f"Full Training Set {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {full_training_results[dataset_name]:.4f}")

       # for method in methods:
           #logging.info(f"{metric.__name__ if hasattr(metric, '__name__') else str(metric)} Improvement ({method}) on {dataset_name}: {result_improvements[method]:.4f}")
    
    return results

Traitement MNIST

In [22]:
def read_images(filename: str) -> np.ndarray:
    """
    Lit les images à partir d'un fichier binaire au format MNIST.
    
    Args:
        filename (str): Le nom du fichier contenant les images.
        
    Returns:
        np.ndarray: Un tableau numpy de forme (nb_images, nb_rows, nb_cols) contenant les images lues.
    
    L'assertion `magic_number == 2051` vérifie que le fichier est bien au format attendu pour les images.
    """
    with open(filename, 'rb') as file:
        # Lire les 16 premiers octets pour obtenir les métadonnées
        magic_number, nb_images, nb_rows, nb_cols = struct.unpack('>IIII', file.read(16))
        
        # Vérification du magic_number (doit être 2051 pour les images MNIST)
        assert magic_number == 2051, "Wrong file format"
        
        # Lire les données d'images et les redimensionner en un tableau 3D (nb_images, nb_rows, nb_cols)
        image_data = np.fromfile(file, dtype=np.uint8).reshape(nb_images, nb_rows, nb_cols)
        
    return image_data

In [23]:
def read_targets(filename: str) -> np.ndarray:
    """
    Lit les cibles (labels) à partir d'un fichier binaire au format MNIST.
    
    Args:
        filename (str): Le nom du fichier contenant les cibles.
        
    Returns:
        np.ndarray: Un tableau numpy contenant les labels des images lues.
    
    L'assertion `magic_number == 2049` vérifie que le fichier est bien au format attendu pour les cibles.
    """
    with open(filename, 'rb') as file:
        # Lire les 8 premiers octets pour obtenir les métadonnées
        magic_number, nb_items = struct.unpack('>II', file.read(8))
        
        # Vérification du magic_number (doit être 2049 pour les cibles MNIST)
        assert magic_number == 2049, "Wrong file format"
        
        # Lire les labels
        targets = np.fromfile(file, dtype=np.uint8)
        
    return targets

In [24]:
path_images_file = '../t10k-images.idx3-ubyte'
path_to_targets = '../t10k-labels.idx1-ubyte'

X_MNIST= read_images(path_images_file).reshape(-1, 28*28)
y_MNIST = read_targets(path_to_targets)
unique_classes, class_counts = np.unique(y_MNIST, return_counts=True)

print(X_MNIST.shape)
for cls, count in zip(unique_classes, class_counts):
    print(f"Classe {cls}: {count} échantillons")

(10000, 784)
Classe 0: 980 échantillons
Classe 1: 1135 échantillons
Classe 2: 1032 échantillons
Classe 3: 1010 échantillons
Classe 4: 982 échantillons
Classe 5: 892 échantillons
Classe 6: 958 échantillons
Classe 7: 1028 échantillons
Classe 8: 974 échantillons
Classe 9: 1009 échantillons


Traitement PRS

In [25]:
color_values = {
    "Yellow": 2,
    "Brown": 2,
    "Red": 2,
    "Black": 2,
    "Grey": 2,
    "Pink": 2,
    "No flag": 0,
    "Blue": 1,
    "Bluish": 1,
    "Purple": 1,
    "Green": 0,
    "Salmon": 1,
}

In [31]:
df1=pd.read_csv("all_reviews_features 2.csv")
df2=pd.read_csv("reviews_since_february_with_features 1.csv")
df3=pd.read_csv("reviews_test_nb_bets 2.csv")
df=pd.concat([df1,df2,df3])

# Charger les datasets
df1 = pd.read_csv("all_reviews_features 2.csv")
df2 = pd.read_csv("reviews_since_february_with_features 1.csv")
df3 = pd.read_csv("reviews_test_nb_bets 2.csv")

# Combinaison des datasets
df = pd.concat([df1, df2, df3])

# Définir les caractéristiques
FEATURES = ['SINGLE_PROPORTION_30D', 'MARKET_MARGIN_30D', 'TURNOVER_PER_BET_30D', 'DEPOSIT_MAX_30D', 
            'SHIELD_REJECTION_30D', 'LIVE_PROPORTION_30D', 'GGR_PER_BET_30D', 'MARGIN_30D', 
            'MARGIN_PER_BET_30D', 'CLOSING_LINE_VALUE_30D', 'MAX_STAKE_RATIO_30D', 'MARKET_TYPE_SCORE_30D', 
            'BET_SCORE_30D', 'TIME_BEFORE_EVENT_30D', 'LATE_BET_ACTION_COUNT_30D', 
            'LATE_BET_STOLEN_AMOUNT_EURO_30D', 'SINGLE_PROPORTION_10D', 'MARKET_MARGIN_10D', 
            'TURNOVER_PER_BET_10D', 'DEPOSIT_MAX_10D', 'SHIELD_REJECTION_10D', 'LIVE_PROPORTION_10D', 
            'GGR_PER_BET_10D', 'MARGIN_10D', 'MARGIN_PER_BET_10D', 'CLOSING_LINE_VALUE_10D', 
            'MAX_STAKE_RATIO_10D', 'MARKET_TYPE_SCORE_10D', 'BET_SCORE_10D', 'TIME_BEFORE_EVENT_10D', 
            'LATE_BET_ACTION_COUNT_10D', 'LATE_BET_STOLEN_AMOUNT_EURO_10D', 'SELECTION_DIVERSITY_LAST_0_DAYS', 
            'SHARP_SHARE_LAST_0_DAYS', 'YELLOW_BREAKOUT_LAST_0_DAYS', 'RISKY_RING_RATIO_LAST_0_DAYS', 
            'SINGLE_PROPORTION_365D', 'MARKET_MARGIN_365D', 'TURNOVER_PER_BET_365D', 'DEPOSIT_MAX_365D', 
            'SHIELD_REJECTION_365D', 'LIVE_PROPORTION_365D', 'GGR_PER_BET_365D', 'MARGIN_PER_BET_365D', 
            'MARGIN_365D', 'CLOSING_LINE_VALUE_365D', 'MARKET_TYPE_SCORE_365D', 'BET_SCORE_365D', 
            'SHARP_SHARE_365D', 'TIME_BEFORE_EVENT_365D', 'MAX_STAKE_RATIO_365D', 'SINGLE_PROPORTION_180D', 
            'MARKET_MARGIN_180D', 'TURNOVER_PER_BET_180D', 'DEPOSIT_MAX_180D', 'SHIELD_REJECTION_180D', 
            'LIVE_PROPORTION_180D', 'GGR_PER_BET_180D', 'MARGIN_PER_BET_180D', 'MARGIN_180D', 
            'CLOSING_LINE_VALUE_180D', 'MARKET_TYPE_SCORE_180D', 'BET_SCORE_180D', 'SHARP_SHARE_180D', 
            'TIME_BEFORE_EVENT_180D', 'MAX_STAKE_RATIO_180D', 'SINGLE_PROPORTION_90D', 'MARKET_MARGIN_90D', 
            'TURNOVER_PER_BET_90D', 'DEPOSIT_MAX_90D', 'SHIELD_REJECTION_90D', 'LIVE_PROPORTION_90D', 
            'GGR_PER_BET_90D', 'MARGIN_PER_BET_90D', 'MARGIN_90D', 'CLOSING_LINE_VALUE_90D', 
            'MARKET_TYPE_SCORE_90D', 'BET_SCORE_90D', 'SHARP_SHARE_90D', 'TIME_BEFORE_EVENT_90D', 
            'MAX_STAKE_RATIO_90D', 'LATE_BET_ACTION_COUNT_90D', 'LATE_BET_STOLEN_AMOUNT_EURO_90D', 
            'SHARP_SHARE_30D', 'SHARP_SHARE_10D', 'has_previous', 'NB_BET_LAST_10_DAYS', 
            'NB_BET_LAST_30_DAYS', 'NB_BET_LAST_90_DAYS', 'NB_BET_LAST_180_DAYS', 'NB_BET_LAST_365_DAYS', 
            'TURNOVER_LAST_10_DAYS', 'TURNOVER_LAST_30_DAYS', 'TURNOVER_LAST_90_DAYS', 
            'TURNOVER_LAST_180_DAYS', 'TURNOVER_LAST_365_DAYS', 'DEPOSIT_LAST_10_DAYS', 
            'DEPOSIT_LAST_30_DAYS', 'DEPOSIT_LAST_90_DAYS', 'DEPOSIT_LAST_180_DAYS', 'DEPOSIT_LAST_365_DAYS', 
            'GGR_LAST_10_DAYS', 'GGR_LAST_30_DAYS', 'GGR_LAST_90_DAYS', 'GGR_LAST_180_DAYS', 
            'GGR_LAST_365_DAYS', 'ORIGINAL_FLAG']

# Mapper les colonnes 'CURRENT_FLAG' et 'ORIGINAL_FLAG' avec color_values
color_values = {
    "Yellow": 2,
    "Brown": 2,
    "Red": 2,
    "Black": 2,
    "Grey": 2,
    "Pink": 2,
    "No flag": 0,
    "Blue": 1,
    "Bluish": 1,
    "Purple": 1,
    "Green": 0,
    "Salmon": 1,
}
df['label'] = df['CURRENT_FLAG'].map(color_values).fillna(0)  # Remplacement et gestion des valeurs manquantes
df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].map(color_values).fillna(0)
df['label'] = df['label'].replace(1, 0)  # Correction spécifique si nécessaire
df['label'] = df['label'].replace(2, 1)  # pour adapter à AUC

df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].replace(1, 0)  # Correction spécifique si nécessaire
df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].replace(2, 1)  # pour adapter à AUC
print(df.columns.get_loc("ORIGINAL_FLAG"),len(FEATURES),FEATURES.index("ORIGINAL_FLAG")
)
# Extraire les features et labels
X_PRS = df[FEATURES]
y_PRS = df["label"]
print(y_PRS.value_counts())
# Conversion en numpy arrays
X_PRS = X_PRS.to_numpy()
y_PRS = y_PRS.to_numpy()

print(df[df['ORIGINAL_FLAG']==1].shape)
print(df[df['ORIGINAL_FLAG']==0].shape)



112 107 106
label
0.0    18373
1.0     3877
Name: count, dtype: int64
(2035, 155)
(20215, 155)


Dataset foot : https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats/data

In [87]:
df=pd.read_csv("2022-2023 Football Player Stats.csv",delimiter=";")
df=df.drop(['Rk','Player','Nation','Squad','Comp'],axis=1)
# Créer une instance de LabelEncoder
label_encoder = LabelEncoder()

# Appliquer fit_transform à la colonne 'Pos' et assigner les valeurs encodées à la nouvelle colonne 'Pos'
df['Pos'] = label_encoder.fit_transform(df['Pos'])

X_foot=df.drop("Pos",axis=1).to_numpy()
y_foot=df["Pos"].to_numpy()

print(np.unique(y_foot,return_counts=True),
X_foot.shape)
# Appliquer SMOTE
#smote = SMOTE(sampling_strategy={0: 959, 1: 950, 2: 939, 3: 910, 4: 909, 5: 910, 6: 909, 7: 910, 8: 889, 9: 895}, random_state=42)
#X_foot, y_foot = smote.fit_resample(X_foot, y_foot)

print(np.unique(y_foot,return_counts=True),
X_foot.shape)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([825,  38, 101, 409,  30, 244, 164, 608,  62, 208])) (2689, 118)
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([825,  38, 101, 409,  30, 244, 164, 608,  62, 208])) (2689, 118)


__MNIST avec 1% initialisation et batch 2%__

In [51]:

labeled_ratio=0.01 # Nombre d'échantillons labellisés initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
n_iterations = 100 # Nombre d'itérations d'Active Learning
batch_ratio =0.02 # Taille du batch d'échantillons ajoutés à chaque itération
#methods=["random", "least_confident", "margin", "entropy","hybrid","qbc-variance","qbc-entropy","qbc-KL"]
methods=["random", "least_confident", "margin", "entropy"]
model_class=lambda : RandomForestClassifier()
# Créer un comité avec 3 modèles différents
# Liste de modèles pour le comité
models = [
    clone(RandomForestClassifier()),  # Random Forest
    clone(LogisticRegression(max_iter=1000)), # Régression Logistique
    clone(SVC(probability=True)),      # Régression Ridge pour classificati)           # Analyse discriminante quadratique
]
#datasets={"MNIST":(X_MNIST,y_MNIST),"PRS":(X_PRS,y_PRS),"Foot":(X_foot,y_foot)}
datasets={"MNIST":(X_MNIST,y_MNIST)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Exécution de l'expérience
run_active_learning_experiment_datasets(datasets, METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models)


Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 100
Nb de données labellisées en plus à chaque itération: 2.00% (160/8000)

Batch réduit à 80 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (random) on MNIST: 0.9521

Batch réduit à 80 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (least_confident) on MNIST: 0.9564

Batch réduit à 80 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (margin) on MNIST: 0.9495

Batch réduit à 80 échantillons car la pool e

{'MNIST': {'random': [0.7196215285501781,
   0.7116630698020001,
   0.8302736906476504,
   0.8583593840379826,
   0.8767697342524947,
   0.8885562994172321,
   0.8941135916764434,
   0.9024275262781938,
   0.9089729799213038,
   0.9132344871583074,
   0.9138294347130786,
   0.9155061490417481,
   0.9202711011194733,
   0.922938818987,
   0.9289186407665581,
   0.9319583374030946,
   0.9244172776250492,
   0.9360094764805454,
   0.9286329455651201,
   0.927609549040255,
   0.9337895894300736,
   0.9388415992123319,
   0.9383384561790434,
   0.9343349104525469,
   0.9368121697107459,
   0.9341762550425778,
   0.936079169854281,
   0.9377505334329082,
   0.9398556396302528,
   0.9421777613227669,
   0.9408019012467935,
   0.9408165150197897,
   0.9419130077994786,
   0.9422238389072783,
   0.9478816168513632,
   0.9429977032164978,
   0.9465301512675813,
   0.9480895466765104,
   0.9474037050267871,
   0.9496654750721497,
   0.9484096707029976,
   0.945524867389907,
   0.9496337381810884,

__PRS avec 1% initialisation et batch 2%__

In [52]:

labeled_ratio=0.01 # Nombre d'échantillons labellisés initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
n_iterations = 100 # Nombre d'itérations d'Active Learning
batch_ratio =0.02 # Taille du batch d'échantillons ajoutés à chaque itération
#methods=["random", "least_confident", "margin", "entropy","hybrid","qbc-variance","qbc-entropy","qbc-KL"]
methods=["random", "least_confident", "margin", "entropy"]
model_class=lambda : RandomForestClassifier()
# Créer un comité avec 3 modèles différents
# Liste de modèles pour le comité
models = [
    clone(RandomForestClassifier()),  # Random Forest
    clone(LogisticRegression(max_iter=1000)), # Régression Logistique
    clone(SVC(probability=True)),      # Régression Ridge pour classificati)           # Analyse discriminante quadratique
]
#datasets={"MNIST":(X_MNIST,y_MNIST),"PRS":(X_PRS,y_PRS),"Foot":(X_foot,y_foot)}
datasets={"PRS":(X_PRS,y_PRS)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Exécution de l'expérience
run_active_learning_experiment_datasets(datasets, METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models)


Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 100
Nb de données labellisées en plus à chaque itération: 2.00% (356/17800)

Batch réduit à 178 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.

Batch réduit à 178 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.

Batch réduit à 178 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.

Batch réduit à 178 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.


{'PRS': {'random': [np.float64(0.5660681347726207),
   np.float64(0.5903846905608706),
   np.float64(0.6499469264542843),
   np.float64(0.6794834751085689),
   np.float64(0.6704184484666582),
   np.float64(0.7120478012067873),
   np.float64(0.7087692459438104),
   np.float64(0.7249395288429226),
   np.float64(0.7213778613913772),
   np.float64(0.7240776057997427),
   np.float64(0.7356878288603825),
   np.float64(0.7383091490638374),
   np.float64(0.7326912305823029),
   np.float64(0.7381390473669525),
   np.float64(0.7327053677696846),
   np.float64(0.7389327353958934),
   np.float64(0.7407764157513226),
   np.float64(0.7528985108956022),
   np.float64(0.7455202890084123),
   np.float64(0.7544066971484291),
   np.float64(0.7547833454180493),
   np.float64(0.7516683418381075),
   np.float64(0.7572211649183967),
   np.float64(0.7622115587106113),
   np.float64(0.76005821703063),
   np.float64(0.7619031406780582),
   np.float64(0.7563556643527201),
   np.float64(0.7657440888418972),
   np

__Foot avec 1% initialisation et batch 2%__

In [55]:

labeled_ratio=0.01 # Nombre d'échantillons labellisés initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
n_iterations = 100 # Nombre d'itérations d'Active Learning
batch_ratio =0.02 # Taille du batch d'échantillons ajoutés à chaque itération
#methods=["random", "least_confident", "margin", "entropy","hybrid","qbc-variance","qbc-entropy","qbc-KL"]
methods=["random", "least_confident", "margin", "entropy"]
model_class=lambda : RandomForestClassifier()
# Créer un comité avec 3 modèles différents
# Liste de modèles pour le comité
models = [
    clone(RandomForestClassifier()),  # Random Forest
    clone(LogisticRegression(max_iter=1000)), # Régression Logistique
    clone(SVC(probability=True)),      # Régression Ridge pour classificati)           # Analyse discriminante quadratique
]
#datasets={"MNIST":(X_MNIST,y_MNIST),"PRS":(X_PRS,y_PRS),"Foot":(X_foot,y_foot)}
datasets={"Foot":(X_foot,y_foot)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Exécution de l'expérience
run_active_learning_experiment_datasets(datasets, METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models)


Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 0.98% (21/2151)
Taille de l'ensemble non-labellisé dans le training set: 99.02% (2130/2151)
Nb d'itérations: 100
Nb de données labellisées en plus à chaque itération: 2.00% (43/2151)

Batch réduit à 23 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (random) on Foot: 0.3940

Batch réduit à 23 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (least_confident) on Foot: 0.4047

Batch réduit à 23 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 50 itérations.
Final f1_score (margin) on Foot: 0.4024

Batch réduit à 23 échantillons car la pool est presque 

{'Foot': {'random': [0.3156388667244261,
   0.33115306171366055,
   0.359755139178638,
   0.3710567272802471,
   0.3827372627372627,
   0.37320322805327344,
   0.3661306454322048,
   0.3537249271594912,
   0.3707789255596443,
   0.36503821149447646,
   0.3902976168979328,
   0.3808384975309923,
   0.4029361055540021,
   0.38037091121302136,
   0.36780023297264675,
   0.3797896979589743,
   0.3986595476135118,
   0.39027865802059347,
   0.38208140008140007,
   0.41926934901051177,
   0.40041312255989076,
   0.3870659790364547,
   0.3921371234462806,
   0.385001135224301,
   0.40011086350300146,
   0.417436135273221,
   0.40128303064416526,
   0.3800228777164779,
   0.3972509314393636,
   0.40692056671421495,
   0.3956074160256199,
   0.3975209404158299,
   0.425536555417826,
   0.3884561693529871,
   0.3817936132554487,
   0.3911951933848526,
   0.38131753312055466,
   0.3970705039023938,
   0.39308026754845626,
   0.3883907355165567,
   0.3825304895948069,
   0.41096292102771537,
   0.

__évolution métrique MNIST en donnant jusqu'à 30% du training set avec différents batch_ratio. 1% initialement labellisé__

In [59]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"MNIST":(X_MNIST,y_MNIST)}

# Liste des batch ratios à tester
batch_ratios = [0.01,0.05,0.1]
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets, METRICS,labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)


Testing batch_ratio = 0.01 with n_iterations = 29


Final f1_score (random) on MNIST: 0.9252
Final f1_score (least_confident) on MNIST: 0.9633
Final f1_score (margin) on MNIST: 0.9633
Final f1_score (entropy) on MNIST: 0.9547



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 6
Nb de données labellisées en plus à chaque itération: 5.00% (400/8000)


Testing batch_ratio = 0.05 with n_iterations = 6


Final f1_score (random) on MNIST: 0.9209
Final f1_score (least_confident) on MNIST: 0.9482
Final f1_score (margin) on MNIST: 0.9555
Final f1_score (entropy) on MNIST: 0.9459



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 3
Nb de données labellisées en plus à chaque itération: 10.00% (800/8000)


Testing batch_ratio = 0.1 with n_iterations = 3


Final f1_score (random) on MNIST: 0.9178
Final f1_score (least_confident) on MNIST: 0.9451
Final f1_score (margin) on MNIST: 0.9326
Final f1_score (entropy) on MNIST: 0.9271


In [464]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"MNIST":(X_MNIST,y_MNIST)}

# Liste des batch ratios à tester
batch_ratios = [0.01,0.02,0.05,0.075,0.1,0.5]
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets, METRICS,labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)


Testing batch_ratio = 0.01 with n_iterations = 29


random - Iteration 1: 2.00% (160/8000) samples labeled, f1_score: 0.7267 on MNIST
random - Iteration 2: 3.00% (240/8000) samples labeled, f1_score: 0.8061 on MNIST
random - Iteration 3: 4.00% (320/8000) samples labeled, f1_score: 0.8321 on MNIST
random - Iteration 4: 5.00% (400/8000) samples labeled, f1_score: 0.8534 on MNIST
random - Iteration 5: 6.00% (480/8000) samples labeled, f1_score: 0.8674 on MNIST
random - Iteration 6: 7.00% (560/8000) samples labeled, f1_score: 0.8752 on MNIST
random - Iteration 7: 8.00% (640/8000) samples labeled, f1_score: 0.8844 on MNIST
random - Iteration 8: 9.00% (720/8000) samples labeled, f1_score: 0.8837 on MNIST
random - Iteration 9: 10.00% (800/8000) samples labeled, f1_score: 0.8934 on MNIST
random - Iteration 10: 11.00% (880/8000) samples labeled, f1_score: 0.8935 on MNIST
random - Iteration 11: 12.00% (960/8000) samples labeled, f1_score: 0.9025 on MNIST
random - Iteration 12: 13.00% (1040/8000) samples labeled, f1_score: 0.9025 on MNIST
random -

Initial f1_score on MNIST: 0.7181
Full Training Set f1_score on MNIST: 0.9547
f1_score Improvement (random) on MNIST: 0.2118
f1_score Improvement (least_confident) on MNIST: 0.2382
f1_score Improvement (margin) on MNIST: 0.2430
f1_score Improvement (entropy) on MNIST: 0.2350

Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 15
Nb de données labellisées en plus à chaque itération: 2.00% (160/8000)


Testing batch_ratio = 0.02 with n_iterations = 15


...

random - Iteration 1: 11.00% (880/8000) samples labeled, f1_score: 0.7217 on MNIST
random - Iteration 2: 21.00% (1680/8000) samples labeled, f1_score: 0.8904 on MNIST
random - Iteration 3: 31.00% (2480/8000) samples labeled, f1_score: 0.9200 on MNIST
Final f1_score (random) on MNIST: 0.9200
least_confident - Iteration 1: 11.00% (880/8000) samples labeled, f1_score: 0.7158 on MNIST
least_confident - Iteration 2: 21.00% (1680/8000) samples labeled, f1_score: 0.8690 on MNIST
least_confident - Iteration 3: 31.00% (2480/8000) samples labeled, f1_score: 0.9363 on MNIST
Final f1_score (least_confident) on MNIST: 0.9363
margin - Iteration 1: 11.00% (880/8000) samples labeled, f1_score: 0.7107 on MNIST
margin - Iteration 2: 21.00% (1680/8000) samples labeled, f1_score: 0.8929 on MNIST
margin - Iteration 3: 31.00% (2480/8000) samples labeled, f1_score: 0.9368 on MNIST
Final f1_score (margin) on MNIST: 0.9368
entropy - Iteration 1: 11.00% (880/8000) samples labeled, f1_score: 0.7198 on MNIST
entr

Initial f1_score on MNIST: 0.7197
Full Training Set f1_score on MNIST: 0.9550
f1_score Improvement (random) on MNIST: 0.2004
f1_score Improvement (least_confident) on MNIST: 0.2166
f1_score Improvement (margin) on MNIST: 0.2171
f1_score Improvement (entropy) on MNIST: 0.2088

Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 1
Nb de données labellisées en plus à chaque itération: 50.00% (4000/8000)


Testing batch_ratio = 0.5 with n_iterations = 1


random - Iteration 1: 51.00% (4080/8000) samples labeled, f1_score: 0.7325 on MNIST
Final f1_score (random) on MNIST: 0.7325
least_confident - Iteration 1: 51.00% (4080/8000) samples labeled, f1_score: 0.7168 on MNIST
Final f1_score (least_confident) on MNIST: 0.7168
margin - Iteration 1: 51.00% (4080/8000) samples labeled, f1_score: 0.7254 on MNIST
Final f1_score (margin) on MNIST: 0.7254
entropy - Iteration 1: 51.00% (4080/8000) samples labeled, f1_score: 0.7126 on MNIST
Final f1_score (entropy) on MNIST: 0.7126


Initial f1_score on MNIST: 0.7137
Full Training Set f1_score on MNIST: 0.9500
f1_score Improvement (random) on MNIST: 0.0188
f1_score Improvement (least_confident) on MNIST: 0.0031
f1_score Improvement (margin) on MNIST: 0.0117
f1_score Improvement (entropy) on MNIST: -0.0011


__évolution métrique PRS en donnant jusqu'à 30% du training set avec différents batch_ratio. 1% initialement labellisé__

In [60]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"PRS":(X_PRS,y_PRS)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Liste des batch ratios à tester
batch_ratios = [0.01,0.05,0.1]

# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets, METRICS ,labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 1.00% (178/17800)


Testing batch_ratio = 0.01 with n_iterations = 29



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 6
Nb de données labellisées en plus à chaque itération: 5.00% (890/17800)


Testing batch_ratio = 0.05 with n_iterations = 6



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 3
Nb de données labellisées en plus à chaque itération: 10.00% (1780/17800)


Testing batch_ratio = 0.1 with n_iterations = 3


In [466]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"PRS":(X_PRS,y_PRS)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Liste des batch ratios à tester
batch_ratios = [0.01,0.02,0.05,0.075,0.1,0.5]

# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets, METRICS ,labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 1.00% (178/17800)


Testing batch_ratio = 0.01 with n_iterations = 29
Classes présentes dans y_train: (array([0., 1.]), array([147,  31]))
Shape de predict_proba: (4450, 2)


random - Iteration 1: 2.00% (356/17800) samples labeled, PR AUC: 0.5868 on PRS
random - Iteration 2: 3.00% (534/17800) samples labeled, PR AUC: 0.6444 on PRS
random - Iteration 3: 4.00% (712/17800) samples labeled, PR AUC: 0.6810 on PRS
random - Iteration 4: 5.00% (890/17800) samples labeled, PR AUC: 0.6775 on PRS
random - Iteration 5: 6.00% (1068/17800) samples labeled, PR AUC: 0.6818 on PRS
random - Iteration 6: 7.00% (1246/17800) samples labeled, PR AUC: 0.6834 on PRS
random - Iteration 7: 8.00% (1424/17800) samples labeled, PR AUC: 0.6878 on PRS
random - Iteration 8: 9.00% (1602/17800) samples labeled, PR AUC: 0.6869 on PRS
random - Iteration 9: 10.00% (1780/17800) samples labeled, PR AUC: 0.7017 on PRS
random - Iteration 10: 11.00% (1958/17800) samples labeled, PR AUC: 0.7114 on PRS
random - Iteration 11: 12.00% (2136/17800) samples labeled, PR AUC: 0.7174 on PRS
random - Iteration 12: 13.00% (2314/17800) samples labeled, PR AUC: 0.7271 on PRS
random - Iteration 13: 14.00% (2492/1

Initial PR AUC on PRS: 0.5845
Full Training Set PR AUC on PRS: 0.7729
PR AUC Improvement (random) on PRS: 0.1760
PR AUC Improvement (least_confident) on PRS: 0.1682
PR AUC Improvement (margin) on PRS: 0.1610
PR AUC Improvement (entropy) on PRS: 0.1675

Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 15
Nb de données labellisées en plus à chaque itération: 2.00% (356/17800)


Testing batch_ratio = 0.02 with n_iterations = 15
Classes présentes dans y_train: (array([0., 1.]), array([147,  31]))
Shape de predict_proba: (4450, 2)


...

random - Iteration 1: 11.00% (1958/17800) samples labeled, PR AUC: 0.5802 on PRS
random - Iteration 2: 21.00% (3738/17800) samples labeled, PR AUC: 0.7105 on PRS
random - Iteration 3: 31.00% (5518/17800) samples labeled, PR AUC: 0.7353 on PRS
Final PR AUC (random) on PRS: 0.7353
least_confident - Iteration 1: 11.00% (1958/17800) samples labeled, PR AUC: 0.5746 on PRS
least_confident - Iteration 2: 21.00% (3738/17800) samples labeled, PR AUC: 0.7237 on PRS
least_confident - Iteration 3: 31.00% (5518/17800) samples labeled, PR AUC: 0.7280 on PRS
Final PR AUC (least_confident) on PRS: 0.7280
margin - Iteration 1: 11.00% (1958/17800) samples labeled, PR AUC: 0.5945 on PRS
margin - Iteration 2: 21.00% (3738/17800) samples labeled, PR AUC: 0.6975 on PRS
margin - Iteration 3: 31.00% (5518/17800) samples labeled, PR AUC: 0.7375 on PRS
Final PR AUC (margin) on PRS: 0.7375
entropy - Iteration 1: 11.00% (1958/17800) samples labeled, PR AUC: 0.5781 on PRS
entropy - Iteration 2: 21.00% (3738/17800)

Initial PR AUC on PRS: 0.5841
Full Training Set PR AUC on PRS: 0.7734
PR AUC Improvement (random) on PRS: 0.1513
PR AUC Improvement (least_confident) on PRS: 0.1439
PR AUC Improvement (margin) on PRS: 0.1534
PR AUC Improvement (entropy) on PRS: 0.1465

Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 1.00% (178/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (17622/17800)
Nb d'itérations: 1
Nb de données labellisées en plus à chaque itération: 50.00% (8900/17800)


Testing batch_ratio = 0.5 with n_iterations = 1
Classes présentes dans y_train: (array([0., 1.]), array([147,  31]))
Shape de predict_proba: (4450, 2)


random - Iteration 1: 51.00% (9078/17800) samples labeled, PR AUC: 0.5605 on PRS
Final PR AUC (random) on PRS: 0.5605
least_confident - Iteration 1: 51.00% (9078/17800) samples labeled, PR AUC: 0.5798 on PRS
Final PR AUC (least_confident) on PRS: 0.5798
margin - Iteration 1: 51.00% (9078/17800) samples labeled, PR AUC: 0.5808 on PRS
Final PR AUC (margin) on PRS: 0.5808
entropy - Iteration 1: 51.00% (9078/17800) samples labeled, PR AUC: 0.5912 on PRS
Final PR AUC (entropy) on PRS: 0.5912


Initial PR AUC on PRS: 0.5937
Full Training Set PR AUC on PRS: 0.7706
PR AUC Improvement (random) on PRS: -0.0332
PR AUC Improvement (least_confident) on PRS: -0.0140
PR AUC Improvement (margin) on PRS: -0.0130
PR AUC Improvement (entropy) on PRS: -0.0026


__évolution métrique Foot en donnant jusqu'à 30% du training set avec différents batch_ratio. 1% initialement labellisé__

In [ ]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"Foot":(X_foot,y_foot)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Liste des batch ratios à tester
batch_ratios = [0.01,0.05,0.1]

# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets,METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 0.98% (21/2151)
Taille de l'ensemble non-labellisé dans le training set: 99.02% (2130/2151)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 0.98% (21/2151)


Testing batch_ratio = 0.01 with n_iterations = 29


Final f1_score (random) on Foot: 0.3986
Final f1_score (least_confident) on Foot: 0.4161
Final f1_score (margin) on Foot: 0.3775
Final f1_score (entropy) on Foot: 0.4227



Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 0.98% (21/2151)
Taille de l'ensemble non-labellisé dans le training set: 99.02% (2130/2151)
Nb d'itérations: 6
Nb de données labellisées en plus à chaque itération: 4.97% (107/2151)


Testing batch_ratio = 0.05 with n_iterations = 6


Final f1_score (random) on Foot: 0.3666
Final f1_score (least_confident) on Foot: 0.4161
Final f1_score (margin) on Foot: 0.3828
Final f1_score (entropy) on Foot: 0.4193



Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 0.98% (21/2151)
Taille de l'ensemble non-labellisé dans le training set: 99.02% (2130/2151)
Nb d'itérations: 3
Nb de données labellisées en plus à chaque itération: 10.00% (215/2151)


Testing batch_ratio = 0.1 with n_iterations = 3


Final f1_score (random) on Foot: 0.3767
Final f1_score (least_confident) on Foot: 0.4035
Final f1_score (margin) on Foot: 0.3901
Final f1_score (entropy) on Foot: 0.3830


In [467]:
# Paramètres fixes
labeled_ratio = 0.01  # 1% de données labellisées initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
methods = ["random", "least_confident", "margin", "entropy"]
model_class = lambda: RandomForestClassifier()
datasets={"Foot":(X_foot,y_foot)}
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}
# Liste des batch ratios à tester
batch_ratios = [0.01,0.02,0.05,0.075,0.1,0.5]

# Stockage des résultats
accuracy_results = {}

for batch_ratio in batch_ratios:
    # Calculer le nombre d'itérations nécessaires pour atteindre 30%
    n_iterations = int(np.ceil((0.30 - labeled_ratio) / batch_ratio))
    
    print(f"Testing batch_ratio = {batch_ratio} with n_iterations = {n_iterations}")
    
    # Exécuter l'expérience
    accuracy_results[batch_ratio] = run_active_learning_experiment_datasets(
        datasets,METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models
    )





Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (9180/9180)
Taille de l'ensemble de test: 20.00% (1836/9180)
Taille de l'ensemble de training: 80.00% (7344/9180)
Taille de l'ensemble labellisé dans le training set: 0.99% (73/7344)
Taille de l'ensemble non-labellisé dans le training set: 99.01% (7271/7344)
Nb d'itérations: 29
Nb de données labellisées en plus à chaque itération: 0.99% (73/7344)


Testing batch_ratio = 0.01 with n_iterations = 29


random - Iteration 1: 1.99% (146/7344) samples labeled, f1_score: 0.5030 on Foot
random - Iteration 2: 2.98% (219/7344) samples labeled, f1_score: 0.5346 on Foot
random - Iteration 3: 3.98% (292/7344) samples labeled, f1_score: 0.5935 on Foot
random - Iteration 4: 4.97% (365/7344) samples labeled, f1_score: 0.6077 on Foot
random - Iteration 5: 5.96% (438/7344) samples labeled, f1_score: 0.6441 on Foot
random - Iteration 6: 6.96% (511/7344) samples labeled, f1_score: 0.6654 on Foot
random - Iteration 7: 7.95% (584/7344) samples labeled, f1_score: 0.6933 on Foot
random - Iteration 8: 8.95% (657/7344) samples labeled, f1_score: 0.6919 on Foot
random - Iteration 9: 9.94% (730/7344) samples labeled, f1_score: 0.7104 on Foot
random - Iteration 10: 10.93% (803/7344) samples labeled, f1_score: 0.7152 on Foot
random - Iteration 11: 11.93% (876/7344) samples labeled, f1_score: 0.7472 on Foot
random - Iteration 12: 12.92% (949/7344) samples labeled, f1_score: 0.7609 on Foot
random - Iteration 13:

Initial f1_score on Foot: 0.5088
Full Training Set f1_score on Foot: 0.9397
f1_score Improvement (random) on Foot: 0.3417
f1_score Improvement (least_confident) on Foot: 0.3926
f1_score Improvement (margin) on Foot: 0.4167
f1_score Improvement (entropy) on Foot: 0.3386

Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (9180/9180)
Taille de l'ensemble de test: 20.00% (1836/9180)
Taille de l'ensemble de training: 80.00% (7344/9180)
Taille de l'ensemble labellisé dans le training set: 0.99% (73/7344)
Taille de l'ensemble non-labellisé dans le training set: 99.01% (7271/7344)
Nb d'itérations: 15
Nb de données labellisées en plus à chaque itération: 1.99% (146/7344)


Testing batch_ratio = 0.02 with n_iterations = 15


...

random - Iteration 1: 10.99% (807/7344) samples labeled, f1_score: 0.5003 on Foot
random - Iteration 2: 20.98% (1541/7344) samples labeled, f1_score: 0.7694 on Foot
random - Iteration 3: 30.98% (2275/7344) samples labeled, f1_score: 0.8104 on Foot
Final f1_score (random) on Foot: 0.8104
least_confident - Iteration 1: 10.99% (807/7344) samples labeled, f1_score: 0.4686 on Foot
least_confident - Iteration 2: 20.98% (1541/7344) samples labeled, f1_score: 0.6698 on Foot
least_confident - Iteration 3: 30.98% (2275/7344) samples labeled, f1_score: 0.7851 on Foot
Final f1_score (least_confident) on Foot: 0.7851
margin - Iteration 1: 10.99% (807/7344) samples labeled, f1_score: 0.4918 on Foot
margin - Iteration 2: 20.98% (1541/7344) samples labeled, f1_score: 0.7317 on Foot
margin - Iteration 3: 30.98% (2275/7344) samples labeled, f1_score: 0.8278 on Foot
Final f1_score (margin) on Foot: 0.8278
entropy - Iteration 1: 10.99% (807/7344) samples labeled, f1_score: 0.4994 on Foot
entropy - Iterati

Initial f1_score on Foot: 0.4951
Full Training Set f1_score on Foot: 0.9393
f1_score Improvement (random) on Foot: 0.3153
f1_score Improvement (least_confident) on Foot: 0.2899
f1_score Improvement (margin) on Foot: 0.3327
f1_score Improvement (entropy) on Foot: 0.2612

Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (9180/9180)
Taille de l'ensemble de test: 20.00% (1836/9180)
Taille de l'ensemble de training: 80.00% (7344/9180)
Taille de l'ensemble labellisé dans le training set: 0.99% (73/7344)
Taille de l'ensemble non-labellisé dans le training set: 99.01% (7271/7344)
Nb d'itérations: 1
Nb de données labellisées en plus à chaque itération: 50.00% (3672/7344)


Testing batch_ratio = 0.5 with n_iterations = 1


random - Iteration 1: 50.99% (3745/7344) samples labeled, f1_score: 0.5164 on Foot
Final f1_score (random) on Foot: 0.5164
least_confident - Iteration 1: 50.99% (3745/7344) samples labeled, f1_score: 0.4844 on Foot
Final f1_score (least_confident) on Foot: 0.4844
margin - Iteration 1: 50.99% (3745/7344) samples labeled, f1_score: 0.4920 on Foot
Final f1_score (margin) on Foot: 0.4920
entropy - Iteration 1: 50.99% (3745/7344) samples labeled, f1_score: 0.5044 on Foot
Final f1_score (entropy) on Foot: 0.5044


Initial f1_score on Foot: 0.4869
Full Training Set f1_score on Foot: 0.9416
f1_score Improvement (random) on Foot: 0.0295
f1_score Improvement (least_confident) on Foot: -0.0025
f1_score Improvement (margin) on Foot: 0.0051
f1_score Improvement (entropy) on Foot: 0.0176


__évolution MNIST avec différents % d'initialisation et batch ratio 1%__

In [681]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"MNIST": (X_MNIST, y_MNIST)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 0.20% (16/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (7984/8000)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)


x_values pour labeled_ratio=0.002 : [0.2, 1.2, 2.1999999999999997, 3.2, 4.2, 5.2, 6.2, 7.200000000000001, 8.200000000000001, 9.2, 10.200000000000001, 11.200000000000001, 12.2, 13.200000000000001, 14.200000000000001, 15.2, 16.2, 17.200000000000003, 18.2, 19.2, 20.200000000000003, 21.2, 22.2, 23.200000000000003, 24.2, 25.2, 26.200000000000003, 27.200000000000003, 28.200000000000003, 29.2, 30.2, 31.2, 32.2, 33.2, 34.2, 35.2, 36.199999999999996, 37.2, 38.2, 39.2, 40.2, 41.2, 42.199999999999996, 43.2, 44.2, 45.2, 46.2, 47.2, 48.199999999999996, 49.2, 50.2, 51.2, 52.2, 53.2, 54.2, 55.2, 56.2, 57.2, 58.199999999999996, 59.199999999999996, 60.199999999999996]


random - Iteration 1: 1.20% (96/8000) samples labeled, f1_score: 0.3939 on MNIST
random - Iteration 2: 2.20% (176/8000) samples labeled, f1_score: 0.7145 on MNIST
random - Iteration 3: 3.20% (256/8000) samples labeled, f1_score: 0.8041 on MNIST
random - Iteration 4: 4.20% (336/8000) samples labeled, f1_score: 0.8363 on MNIST
random - Iteration 5: 5.20% (416/8000) samples labeled, f1_score: 0.8475 on MNIST
random - Iteration 6: 6.20% (496/8000) samples labeled, f1_score: 0.8590 on MNIST
random - Iteration 7: 7.20% (576/8000) samples labeled, f1_score: 0.8740 on MNIST
random - Iteration 8: 8.20% (656/8000) samples labeled, f1_score: 0.8901 on MNIST
random - Iteration 9: 9.20% (736/8000) samples labeled, f1_score: 0.8946 on MNIST
random - Iteration 10: 10.20% (816/8000) samples labeled, f1_score: 0.8949 on MNIST
random - Iteration 11: 11.20% (896/8000) samples labeled, f1_score: 0.8955 on MNIST
random - Iteration 12: 12.20% (976/8000) samples labeled, f1_score: 0.8924 on MNIST
random - It

{'MNIST': {'random': [0.4003951550116637, 0.39394638028183715, 0.7145334588726018, 0.8040793534130494, 0.8362717705703769, 0.8475152493338138, 0.858990842742722, 0.8740218614505009, 0.8901400448181563, 0.8945670687927672, 0.8948638471014425, 0.8955335339509322, 0.8923632343762969, 0.9089903753862492, 0.9016273459608259, 0.9047514818240936, 0.9099269760512664, 0.9052961363567305, 0.9140892291078082, 0.9097196604907225, 0.91864753995871, 0.9161506189241324, 0.9159454231950616, 0.9223963259963955, 0.9197407189310747, 0.9235271898638404, 0.9197358553045232, 0.9234455529159975, 0.9249264649036301, 0.9265762502720317, 0.9303584632332204, 0.9268922938005684, 0.9295605862294269, 0.9276754616686768, 0.9302275858504281, 0.9338546506840965, 0.9315769961120098, 0.9372808646059783, 0.9357856355929697, 0.933540791970565, 0.9379004685032335, 0.9341645416376781, 0.9296034922241608, 0.9413042929166897, 0.9387495340728818, 0.9377874559783861, 0.9404235635934745, 0.9352014283494958, 0.9405254879262355, 0

random - Iteration 1: 2.00% (160/8000) samples labeled, f1_score: 0.7230 on MNIST
random - Iteration 2: 3.00% (240/8000) samples labeled, f1_score: 0.7910 on MNIST
random - Iteration 3: 4.00% (320/8000) samples labeled, f1_score: 0.8379 on MNIST
random - Iteration 4: 5.00% (400/8000) samples labeled, f1_score: 0.8524 on MNIST
random - Iteration 5: 6.00% (480/8000) samples labeled, f1_score: 0.8589 on MNIST
random - Iteration 6: 7.00% (560/8000) samples labeled, f1_score: 0.8703 on MNIST
random - Iteration 7: 8.00% (640/8000) samples labeled, f1_score: 0.8770 on MNIST
random - Iteration 8: 9.00% (720/8000) samples labeled, f1_score: 0.8916 on MNIST
random - Iteration 9: 10.00% (800/8000) samples labeled, f1_score: 0.8847 on MNIST
random - Iteration 10: 11.00% (880/8000) samples labeled, f1_score: 0.8977 on MNIST
random - Iteration 11: 12.00% (960/8000) samples labeled, f1_score: 0.8895 on MNIST
random - Iteration 12: 13.00% (1040/8000) samples labeled, f1_score: 0.8933 on MNIST
random -

...

{'MNIST': {'random': [0.9178203345667232, 0.9178511534210252, 0.9205426321145552, 0.918383584912506, 0.9182300037480442, 0.9223057026904362, 0.9270250728993569, 0.9249532436693663, 0.9276710793942655, 0.9254371276318961, 0.9263663455662476, 0.9352029887626362, 0.9259545732360868, 0.931706233328551, 0.9302077286765179, 0.9280974374965588, 0.932246270551973, 0.9332474261095669, 0.9326976726988203, 0.9354225168040837, 0.9312664559777281, 0.9370219313135308, 0.9374833718833795, 0.9362202358324989, 0.9418798082011749, 0.9357666751205939, 0.9410126799682018, 0.9378588460240707, 0.9404418745804277, 0.9407730942517125, 0.9409818927414626, 0.9379946901032346, 0.9500631346928934, 0.9378019572157031, 0.947603447756929, 0.9419778583951459, 0.9457096498566486, 0.934775872130434, 0.9424929483536323, 0.9460232200407145, 0.9423170143751485, 0.9378951061674329], 'margin': [0.9178203345667232, 0.9186402282875618, 0.9292887015538289, 0.931703574565143, 0.934445375453274, 0.9384982030646942, 0.94404775984

random - Iteration 1: 31.00% (2480/8000) samples labeled, f1_score: 0.9284 on MNIST
random - Iteration 2: 32.00% (2560/8000) samples labeled, f1_score: 0.9307 on MNIST
random - Iteration 3: 33.00% (2640/8000) samples labeled, f1_score: 0.9276 on MNIST
random - Iteration 4: 34.00% (2720/8000) samples labeled, f1_score: 0.9359 on MNIST
random - Iteration 5: 35.00% (2800/8000) samples labeled, f1_score: 0.9284 on MNIST
random - Iteration 6: 36.00% (2880/8000) samples labeled, f1_score: 0.9351 on MNIST
random - Iteration 7: 37.00% (2960/8000) samples labeled, f1_score: 0.9280 on MNIST
random - Iteration 8: 38.00% (3040/8000) samples labeled, f1_score: 0.9344 on MNIST
random - Iteration 9: 39.00% (3120/8000) samples labeled, f1_score: 0.9342 on MNIST
random - Iteration 10: 40.00% (3200/8000) samples labeled, f1_score: 0.9297 on MNIST
random - Iteration 11: 41.00% (3280/8000) samples labeled, f1_score: 0.9380 on MNIST
random - Iteration 12: 42.00% (3360/8000) samples labeled, f1_score: 0.932

{'MNIST': {'random': [0.9320459423983083, 0.9284356983475908, 0.9306779539583033, 0.9275640891244123, 0.9358580203061916, 0.9283755020908625, 0.9351090123951898, 0.9279848128944229, 0.9343728413602506, 0.9342270872547731, 0.9297141573436043, 0.9380308858754883, 0.9322857162524649, 0.9399184819673551, 0.9356882153938646, 0.9357139631064413, 0.9404015396605147, 0.9455593868374915, 0.944870607378269, 0.9397678398496092, 0.9433844570682058, 0.9414517142533437, 0.9407828919030197, 0.9435256279975668, 0.9475758126531302, 0.9425025587436944, 0.9399976778466991, 0.9424004189690749, 0.9479503553080126, 0.9434007899825886, 0.9469213674820626, 0.9424307439151807, 0.9425369631309689], 'margin': [0.9320459423983083, 0.926499851157307, 0.9357901139352123, 0.9335184252958235, 0.9399798547896447, 0.9389737577791302, 0.9476803101193623, 0.9407600055392757, 0.9520603401228014, 0.9473119526578258, 0.9529226815180497, 0.9536489460503509, 0.9550409389781004, 0.9574682878126582, 0.9539709499556597, 0.956151

random - Iteration 1: 41.00% (3280/8000) samples labeled, f1_score: 0.9362 on MNIST
random - Iteration 2: 42.00% (3360/8000) samples labeled, f1_score: 0.9374 on MNIST
random - Iteration 3: 43.00% (3440/8000) samples labeled, f1_score: 0.9341 on MNIST
random - Iteration 4: 44.00% (3520/8000) samples labeled, f1_score: 0.9415 on MNIST
random - Iteration 5: 45.00% (3600/8000) samples labeled, f1_score: 0.9378 on MNIST
random - Iteration 6: 46.00% (3680/8000) samples labeled, f1_score: 0.9382 on MNIST
random - Iteration 7: 47.00% (3760/8000) samples labeled, f1_score: 0.9404 on MNIST
random - Iteration 8: 48.00% (3840/8000) samples labeled, f1_score: 0.9367 on MNIST
random - Iteration 9: 49.00% (3920/8000) samples labeled, f1_score: 0.9382 on MNIST
random - Iteration 10: 50.00% (4000/8000) samples labeled, f1_score: 0.9372 on MNIST
random - Iteration 11: 51.00% (4080/8000) samples labeled, f1_score: 0.9455 on MNIST
random - Iteration 12: 52.00% (4160/8000) samples labeled, f1_score: 0.942

{'MNIST': {'random': [0.9356021526055331, 0.9362322243779777, 0.9373538660368542, 0.9341191779736555, 0.9414745217371102, 0.9377847924820036, 0.9381563048744542, 0.9403838022571829, 0.9367286743908902, 0.9381914987034445, 0.9371544612808581, 0.9455337553000002, 0.9421631702943476, 0.9460624387001151, 0.9423923250136544, 0.939909538508384, 0.9387788196666959, 0.9452072620250135, 0.9428766000964421, 0.9484622013282111, 0.9458398978633993, 0.9437744834944353], 'margin': [0.9356021526055331, 0.9316338933596974, 0.9409042789022788, 0.9379575916787724, 0.9444473945361394, 0.9464155414808328, 0.9509483464896125, 0.955695639165574, 0.9545247978862639, 0.9510385001044304, 0.9571833641668326, 0.9504315441480532, 0.9509804229201156, 0.9546045261707654, 0.9567418585246775, 0.9566805145797563, 0.9591543179912112, 0.9560865010329959, 0.9592580366389036, 0.9571027520106217, 0.9561152836258854, 0.9565324171638435]}}
len(y_values_per_iteration)=22 pour random
len(y_values_per_iteration) après ajustemen

__évolution PRS avec différents % d'initialisation et batch ratio 1%__

In [703]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"PRS": (X_PRS, y_PRS)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 0.20% (35/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (17765/17800)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (178/17800)


x_values pour labeled_ratio=0.002 : [0.2, 1.2, 2.1999999999999997, 3.2, 4.2, 5.2, 6.2, 7.200000000000001, 8.200000000000001, 9.2, 10.200000000000001, 11.200000000000001, 12.2, 13.200000000000001, 14.200000000000001, 15.2, 16.2, 17.200000000000003, 18.2, 19.2, 20.200000000000003, 21.2, 22.2, 23.200000000000003, 24.2, 25.2, 26.200000000000003, 27.200000000000003, 28.200000000000003, 29.2, 30.2, 31.2, 32.2, 33.2, 34.2, 35.2, 36.199999999999996, 37.2, 38.2, 39.2, 40.2, 41.2, 42.199999999999996, 43.2, 44.2, 45.2, 46.2, 47.2, 48.199999999999996, 49.2, 50.2, 51.2, 52.2, 53.2, 54.2, 55.2, 56.2, 57.2, 58.199999999999996, 59.199999999999996, 60.199999999999996]
Classes présentes dans y_train: (array([0., 1.]), array([29,  6]))
Shape de predict_proba: (4450, 2)


random - Iteration 1: 1.20% (213/17800) samples labeled, PR AUC: 0.3132 on PRS
random - Iteration 2: 2.20% (391/17800) samples labeled, PR AUC: 0.5885 on PRS
random - Iteration 3: 3.20% (569/17800) samples labeled, PR AUC: 0.6207 on PRS
random - Iteration 4: 4.20% (747/17800) samples labeled, PR AUC: 0.6326 on PRS
random - Iteration 5: 5.20% (925/17800) samples labeled, PR AUC: 0.6437 on PRS
random - Iteration 6: 6.20% (1103/17800) samples labeled, PR AUC: 0.6575 on PRS
random - Iteration 7: 7.20% (1281/17800) samples labeled, PR AUC: 0.6733 on PRS
random - Iteration 8: 8.20% (1459/17800) samples labeled, PR AUC: 0.6710 on PRS
random - Iteration 9: 9.20% (1637/17800) samples labeled, PR AUC: 0.6745 on PRS
random - Iteration 10: 10.20% (1815/17800) samples labeled, PR AUC: 0.6933 on PRS
random - Iteration 11: 11.20% (1993/17800) samples labeled, PR AUC: 0.6953 on PRS
random - Iteration 12: 12.20% (2171/17800) samples labeled, PR AUC: 0.6947 on PRS
random - Iteration 13: 13.20% (2349/178

{'PRS': {'random': [np.float64(0.35515084845100475), np.float64(0.31321762914828105), np.float64(0.5885344261386873), np.float64(0.6207181457713866), np.float64(0.6326192493993802), np.float64(0.6437308065973169), np.float64(0.6574869062118244), np.float64(0.673334596259803), np.float64(0.6710422629194734), np.float64(0.6744601258682992), np.float64(0.6932553384105605), np.float64(0.6953024619433615), np.float64(0.6947104030352522), np.float64(0.6910272626299342), np.float64(0.6927122426175887), np.float64(0.6957382526496209), np.float64(0.6976017134791058), np.float64(0.7105611072367615), np.float64(0.7064254219673974), np.float64(0.710415962498383), np.float64(0.7188971102937044), np.float64(0.7127384079681232), np.float64(0.7248635897317159), np.float64(0.720344791780739), np.float64(0.7241109388034672), np.float64(0.7251718285091551), np.float64(0.7307969505228649), np.float64(0.7288165484316871), np.float64(0.7326444890589912), np.float64(0.7358582166030323), np.float64(0.72788388

random - Iteration 1: 2.00% (356/17800) samples labeled, PR AUC: 0.5835 on PRS
random - Iteration 2: 3.00% (534/17800) samples labeled, PR AUC: 0.6219 on PRS
random - Iteration 3: 4.00% (712/17800) samples labeled, PR AUC: 0.6542 on PRS
random - Iteration 4: 5.00% (890/17800) samples labeled, PR AUC: 0.6545 on PRS
random - Iteration 5: 6.00% (1068/17800) samples labeled, PR AUC: 0.6641 on PRS
random - Iteration 6: 7.00% (1246/17800) samples labeled, PR AUC: 0.6726 on PRS
random - Iteration 7: 8.00% (1424/17800) samples labeled, PR AUC: 0.6802 on PRS
random - Iteration 8: 9.00% (1602/17800) samples labeled, PR AUC: 0.6822 on PRS
random - Iteration 9: 10.00% (1780/17800) samples labeled, PR AUC: 0.7076 on PRS
random - Iteration 10: 11.00% (1958/17800) samples labeled, PR AUC: 0.7115 on PRS
random - Iteration 11: 12.00% (2136/17800) samples labeled, PR AUC: 0.7030 on PRS
random - Iteration 12: 13.00% (2314/17800) samples labeled, PR AUC: 0.7016 on PRS
random - Iteration 13: 14.00% (2492/1

...

{'PRS': {'random': [np.float64(0.7305908327032437), np.float64(0.7325269256767204), np.float64(0.7254916897594333), np.float64(0.7235747638549116), np.float64(0.7298641745674316), np.float64(0.7240821144411527), np.float64(0.7222024306846604), np.float64(0.7276554300957749), np.float64(0.7228409370674329), np.float64(0.730761131659286), np.float64(0.7356113552838328), np.float64(0.7318595332612107), np.float64(0.7411961748331475), np.float64(0.7261322307137759), np.float64(0.7321564721183256), np.float64(0.7261436048637531), np.float64(0.7363701235996203), np.float64(0.746555218306706), np.float64(0.7307670117582722), np.float64(0.7313728775543478), np.float64(0.7284812856998683), np.float64(0.732692165426051), np.float64(0.7421569650222337), np.float64(0.742514821777783), np.float64(0.7454213724920797), np.float64(0.7395788370481187), np.float64(0.7418374064986032), np.float64(0.7517627734716471), np.float64(0.7511746585192269), np.float64(0.7486912087164199), np.float64(0.75317795060

random - Iteration 1: 31.00% (5518/17800) samples labeled, PR AUC: 0.7521 on PRS
random - Iteration 2: 32.00% (5696/17800) samples labeled, PR AUC: 0.7445 on PRS
random - Iteration 3: 33.00% (5874/17800) samples labeled, PR AUC: 0.7493 on PRS
random - Iteration 4: 34.00% (6052/17800) samples labeled, PR AUC: 0.7535 on PRS
random - Iteration 5: 35.00% (6230/17800) samples labeled, PR AUC: 0.7524 on PRS
random - Iteration 6: 36.00% (6408/17800) samples labeled, PR AUC: 0.7489 on PRS
random - Iteration 7: 37.00% (6586/17800) samples labeled, PR AUC: 0.7505 on PRS
random - Iteration 8: 38.00% (6764/17800) samples labeled, PR AUC: 0.7547 on PRS
random - Iteration 9: 39.00% (6942/17800) samples labeled, PR AUC: 0.7531 on PRS
random - Iteration 10: 40.00% (7120/17800) samples labeled, PR AUC: 0.7493 on PRS
random - Iteration 11: 41.00% (7298/17800) samples labeled, PR AUC: 0.7560 on PRS
random - Iteration 12: 42.00% (7476/17800) samples labeled, PR AUC: 0.7550 on PRS
random - Iteration 13: 43

{'PRS': {'random': [np.float64(0.7472942485217613), np.float64(0.752142334033338), np.float64(0.744488886081614), np.float64(0.7492849119511313), np.float64(0.7534877101252786), np.float64(0.7523517899899486), np.float64(0.748877302977251), np.float64(0.750465582508442), np.float64(0.7547110578639117), np.float64(0.7530906100413697), np.float64(0.7492765237006745), np.float64(0.7560459286954155), np.float64(0.7550149954293462), np.float64(0.752187298400216), np.float64(0.75035370832035), np.float64(0.7535947168047805), np.float64(0.7500605493914283), np.float64(0.7544460580134256), np.float64(0.7556939515482719), np.float64(0.7578020876775156), np.float64(0.7559074475425206), np.float64(0.7608204382140225), np.float64(0.7622306515466273), np.float64(0.7565976257004703), np.float64(0.75591607217296), np.float64(0.7567654994632717), np.float64(0.7684372039070984), np.float64(0.7613916456704948), np.float64(0.7608854866486744), np.float64(0.7552789928861701), np.float64(0.7622575416272783

random - Iteration 1: 41.00% (7298/17800) samples labeled, PR AUC: 0.7517 on PRS
random - Iteration 2: 42.00% (7476/17800) samples labeled, PR AUC: 0.7535 on PRS
random - Iteration 3: 43.00% (7654/17800) samples labeled, PR AUC: 0.7569 on PRS
random - Iteration 4: 44.00% (7832/17800) samples labeled, PR AUC: 0.7516 on PRS
random - Iteration 5: 45.00% (8010/17800) samples labeled, PR AUC: 0.7526 on PRS
random - Iteration 6: 46.00% (8188/17800) samples labeled, PR AUC: 0.7521 on PRS
random - Iteration 7: 47.00% (8366/17800) samples labeled, PR AUC: 0.7571 on PRS
random - Iteration 8: 48.00% (8544/17800) samples labeled, PR AUC: 0.7492 on PRS
random - Iteration 9: 49.00% (8722/17800) samples labeled, PR AUC: 0.7541 on PRS
random - Iteration 10: 50.00% (8900/17800) samples labeled, PR AUC: 0.7525 on PRS
random - Iteration 11: 51.00% (9078/17800) samples labeled, PR AUC: 0.7497 on PRS
random - Iteration 12: 52.00% (9256/17800) samples labeled, PR AUC: 0.7554 on PRS
random - Iteration 13: 53

{'PRS': {'random': [np.float64(0.749413837061305), np.float64(0.7517242045777651), np.float64(0.7534603667053752), np.float64(0.7568798853596937), np.float64(0.7515830314168551), np.float64(0.7526482955885161), np.float64(0.7520998858291026), np.float64(0.7571255399334849), np.float64(0.7491833883959464), np.float64(0.754110600968213), np.float64(0.7524995006859052), np.float64(0.7496865579827139), np.float64(0.7553969870089212), np.float64(0.75369180873532), np.float64(0.7566706540370873), np.float64(0.7488426979332311), np.float64(0.7545859302561806), np.float64(0.7593050086396301), np.float64(0.7543052421188017), np.float64(0.7455351683682563), np.float64(0.7571339338288234), np.float64(0.7562221111137173)], 'margin': [np.float64(0.749413837061305), np.float64(0.7458958172372483), np.float64(0.7535971766360632), np.float64(0.7583320120915242), np.float64(0.757121111612463), np.float64(0.7500784694654786), np.float64(0.7573821286348059), np.float64(0.7610459154989135), np.float64(0.7

__évolution Foot avec différents % d'initialisation et batch ratio 1%__

In [89]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.005,0.3]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"Foot": (X_foot, y_foot)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 0.46% (10/2151)
Taille de l'ensemble non-labellisé dans le training set: 99.54% (2141/2151)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 0.98% (21/2151)


x_values pour labeled_ratio=0.005 : [0.5, 1.5, 2.5, 3.4999999999999996, 4.5, 5.5, 6.5, 7.500000000000001, 8.5, 9.5, 10.500000000000002, 11.5, 12.5, 13.5, 14.500000000000002, 15.5, 16.5, 17.5, 18.5, 19.5, 20.5, 21.5, 22.5, 23.5, 24.5, 25.5, 26.5, 27.500000000000004, 28.500000000000004, 29.5, 30.5, 31.5, 32.5, 33.5, 34.5, 35.50000000000001, 36.5, 37.5, 38.5, 39.5, 40.5, 41.5, 42.5, 43.5, 44.5, 45.5, 46.5, 47.5, 48.5, 49.5, 50.5, 51.5, 52.5, 53.5, 54.50000000000001, 55.50000000000001, 56.50000000000001, 57.50000000000001, 58.5, 59.5, 60.5]


Final f1_score (random) on Foot: 0.4162
Final f1_score (margin) on Foot: 0.3835



Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (2689/2689)
Taille de l'ensemble de test: 20.01% (538/2689)
Taille de l'ensemble de training: 79.99% (2151/2689)
Taille de l'ensemble labellisé dans le training set: 29.99% (645/2151)
Taille de l'ensemble non-labellisé dans le training set: 70.01% (1506/2151)
Nb d'itérations: 32
Nb de données labellisées en plus à chaque itération: 0.98% (21/2151)


{'Foot': {'random': [0.2703053239616086, 0.3080024342507845, 0.33649228770686423, 0.3344756496047914, 0.3222261305537792, 0.3345627830070007, 0.3382523228897994, 0.3320101328590293, 0.3351057160204218, 0.341643126889581, 0.35579523306874405, 0.3451947258836645, 0.34924050624412806, 0.3715160765160765, 0.3817779884908455, 0.3715487081681038, 0.3659445840261748, 0.36448220152323224, 0.3841118361123918, 0.4117684241807235, 0.3892014725378902, 0.3988602585008119, 0.40994750342450414, 0.39482775488548855, 0.3957569064499212, 0.3937579294985223, 0.3737201423822807, 0.38793672877464846, 0.4010407055470654, 0.3923850705659793, 0.40135466825313115, 0.3858766603148651, 0.39854305913504595, 0.37163409959370364, 0.3693646184372329, 0.4026577265327266, 0.3839816860085994, 0.36831268239869963, 0.39337332562199745, 0.3810940512682544, 0.40319681193375895, 0.3944178747448768, 0.41008844097500763, 0.42966802157977535, 0.41094599959793765, 0.4104609085181131, 0.39616402116402116, 0.4000294441368397, 0.4

Final f1_score (random) on Foot: 0.3990
Final f1_score (margin) on Foot: 0.4025


{'Foot': {'random': [0.40144085723325346, 0.41248429289387384, 0.38354286400958376, 0.3993344572795961, 0.3977179606898268, 0.3975043390751683, 0.40505231880813647, 0.38806165612763527, 0.40496671703024534, 0.4042360334387153, 0.38885938799547504, 0.3842986529484732, 0.4064818492617122, 0.412939409534431, 0.41801828514193257, 0.4023296612461283, 0.4037953947598016, 0.39879138511311024, 0.41543983336358375, 0.3977057857360649, 0.39902484325793847, 0.4054603315756805, 0.402519312806222, 0.3998664247629765, 0.4108155812068131, 0.4076901304740109, 0.39528438498011803, 0.40757009816978035, 0.3949342152018723, 0.39010506628920955, 0.4093683276893998, 0.4012189337183251, 0.39901635314369055], 'margin': [0.40144085723325346, 0.3957834808522218, 0.3843598094526212, 0.4142619450749823, 0.3957819203589548, 0.4025047512922864, 0.3973147266787532, 0.40190710139360136, 0.39397637859726053, 0.4166451569970814, 0.37677504718391297, 0.40427618441307356, 0.3978228152007798, 0.3924885179017455, 0.3946742

In [682]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"Foot": (X_foot, y_foot)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: Foot

Métrique utilisée: f1_score
Taille totale du dataset Foot: 100.00% (9180/9180)
Taille de l'ensemble de test: 20.00% (1836/9180)
Taille de l'ensemble de training: 80.00% (7344/9180)
Taille de l'ensemble labellisé dans le training set: 0.19% (14/7344)
Taille de l'ensemble non-labellisé dans le training set: 99.81% (7330/7344)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 0.99% (73/7344)


x_values pour labeled_ratio=0.002 : [0.2, 1.2, 2.1999999999999997, 3.2, 4.2, 5.2, 6.2, 7.200000000000001, 8.200000000000001, 9.2, 10.200000000000001, 11.200000000000001, 12.2, 13.200000000000001, 14.200000000000001, 15.2, 16.2, 17.200000000000003, 18.2, 19.2, 20.200000000000003, 21.2, 22.2, 23.200000000000003, 24.2, 25.2, 26.200000000000003, 27.200000000000003, 28.200000000000003, 29.2, 30.2, 31.2, 32.2, 33.2, 34.2, 35.2, 36.199999999999996, 37.2, 38.2, 39.2, 40.2, 41.2, 42.199999999999996, 43.2, 44.2, 45.2, 46.2, 47.2, 48.199999999999996, 49.2, 50.2, 51.2, 52.2, 53.2, 54.2, 55.2, 56.2, 57.2, 58.199999999999996, 59.199999999999996, 60.199999999999996]


random - Iteration 1: 1.18% (87/7344) samples labeled, f1_score: 0.2980 on Foot
random - Iteration 2: 2.18% (160/7344) samples labeled, f1_score: 0.4629 on Foot
random - Iteration 3: 3.17% (233/7344) samples labeled, f1_score: 0.5627 on Foot
random - Iteration 4: 4.17% (306/7344) samples labeled, f1_score: 0.5974 on Foot
random - Iteration 5: 5.16% (379/7344) samples labeled, f1_score: 0.6419 on Foot
random - Iteration 6: 6.15% (452/7344) samples labeled, f1_score: 0.6672 on Foot
random - Iteration 7: 7.15% (525/7344) samples labeled, f1_score: 0.6852 on Foot
random - Iteration 8: 8.14% (598/7344) samples labeled, f1_score: 0.6857 on Foot
random - Iteration 9: 9.14% (671/7344) samples labeled, f1_score: 0.6973 on Foot
random - Iteration 10: 10.13% (744/7344) samples labeled, f1_score: 0.7317 on Foot
random - Iteration 11: 11.12% (817/7344) samples labeled, f1_score: 0.7261 on Foot
random - Iteration 12: 12.12% (890/7344) samples labeled, f1_score: 0.7444 on Foot
random - Iteration 13: 

{'Foot': {'random': [0.31844851514051054, 0.2980455954146956, 0.462904396833539, 0.5626803106085714, 0.5974184636586263, 0.6418821937572003, 0.6671920418203505, 0.685199478060543, 0.6857242137070992, 0.6972864651210455, 0.7316502055234302, 0.726085356622204, 0.7443568635195851, 0.7488481809722007, 0.756554088875141, 0.764785849586119, 0.7628884301694676, 0.7745661171487483, 0.7808916049888724, 0.7967293988525934, 0.807511631220431, 0.8102304286800581, 0.8215971888707925, 0.8270051694017704, 0.8316139279439613, 0.8369025104850266, 0.8389287421734071, 0.8335611738317923, 0.8431609084030572, 0.8467694425821497, 0.8377074703303183, 0.847594071547908, 0.8537538052951144, 0.8496259688293837, 0.8513697382886354, 0.8651759983547791, 0.8619950864003727, 0.8691979642381357, 0.8625421571974285, 0.8646662329383836, 0.8676174374073125, 0.8669818427907492, 0.8742496954076767, 0.8711224284654469, 0.8857148129904466, 0.8812061762310199, 0.8829309871934518, 0.8815376380080556, 0.8874939305498681, 0.897

random - Iteration 1: 1.99% (146/7344) samples labeled, f1_score: 0.5054 on Foot
random - Iteration 2: 2.98% (219/7344) samples labeled, f1_score: 0.5195 on Foot
random - Iteration 3: 3.98% (292/7344) samples labeled, f1_score: 0.5730 on Foot
random - Iteration 4: 4.97% (365/7344) samples labeled, f1_score: 0.6329 on Foot
random - Iteration 5: 5.96% (438/7344) samples labeled, f1_score: 0.6640 on Foot
random - Iteration 6: 6.96% (511/7344) samples labeled, f1_score: 0.6862 on Foot
random - Iteration 7: 7.95% (584/7344) samples labeled, f1_score: 0.6911 on Foot
random - Iteration 8: 8.95% (657/7344) samples labeled, f1_score: 0.7359 on Foot
random - Iteration 9: 9.94% (730/7344) samples labeled, f1_score: 0.7376 on Foot
random - Iteration 10: 10.93% (803/7344) samples labeled, f1_score: 0.7456 on Foot
random - Iteration 11: 11.93% (876/7344) samples labeled, f1_score: 0.7551 on Foot
random - Iteration 12: 12.92% (949/7344) samples labeled, f1_score: 0.7748 on Foot
random - Iteration 13:

...

{'Foot': {'random': [0.8061688530325272, 0.8053446185276378, 0.8044898150534724, 0.8157012335689707, 0.8184965060308276, 0.8226718766382592, 0.8290022057890031, 0.8378928529879731, 0.8268124840591812, 0.843708066791169, 0.8583547075227239, 0.852201262515355, 0.8541075625134923, 0.869611238737499, 0.8565861560587381, 0.8636183853463709, 0.8719115333575645, 0.8796483444587665, 0.873890702847383, 0.8734441785244739, 0.8816339482923956, 0.8795768326685547, 0.8883302153992234, 0.8791349902045228, 0.8814907553441836, 0.892294507711977, 0.8897606908798867, 0.8880510677032636, 0.8909101276693328, 0.8906593263010543, 0.8921033802005596, 0.9010853598263507, 0.8998710023457811, 0.9036835897901543, 0.8995100712112072, 0.8977528663494436, 0.900679025453336, 0.9019271534194362, 0.89863108127023, 0.9081590887572426, 0.9062858542411545, 0.9091437934582904], 'margin': [0.8061688530325272, 0.7999305360141029, 0.8171012229237616, 0.8394799573016994, 0.8574708301095709, 0.8605303346532036, 0.8773643655158

random - Iteration 1: 30.99% (2276/7344) samples labeled, f1_score: 0.8509 on Foot
random - Iteration 2: 31.99% (2349/7344) samples labeled, f1_score: 0.8405 on Foot
random - Iteration 3: 32.98% (2422/7344) samples labeled, f1_score: 0.8524 on Foot
random - Iteration 4: 33.97% (2495/7344) samples labeled, f1_score: 0.8474 on Foot
random - Iteration 5: 34.97% (2568/7344) samples labeled, f1_score: 0.8529 on Foot
random - Iteration 6: 35.96% (2641/7344) samples labeled, f1_score: 0.8670 on Foot
random - Iteration 7: 36.96% (2714/7344) samples labeled, f1_score: 0.8614 on Foot
random - Iteration 8: 37.95% (2787/7344) samples labeled, f1_score: 0.8564 on Foot
random - Iteration 9: 38.94% (2860/7344) samples labeled, f1_score: 0.8646 on Foot
random - Iteration 10: 39.94% (2933/7344) samples labeled, f1_score: 0.8678 on Foot
random - Iteration 11: 40.93% (3006/7344) samples labeled, f1_score: 0.8767 on Foot
random - Iteration 12: 41.93% (3079/7344) samples labeled, f1_score: 0.8783 on Foot
r

{'Foot': {'random': [0.845094647473231, 0.8508785085081625, 0.8405041975325108, 0.8523652181605419, 0.8474133032671227, 0.8529315905261743, 0.8670040494901616, 0.861362315049444, 0.856384892385629, 0.864550075919556, 0.8678448311733039, 0.8767227997079866, 0.8782724998051096, 0.8764570899467227, 0.8775383807870331, 0.8885946466796767, 0.8821813342445362, 0.8876234699949865, 0.8863911596826333, 0.8825980025390848, 0.8875049188404475, 0.8922038549334937, 0.8837546449961383, 0.8936056551122077, 0.8980084675449627, 0.8947549725889041, 0.8958143152692785, 0.8992382327149991, 0.8985572692576186, 0.8968928381020916, 0.9003985771285817, 0.8944279232013095, 0.9012370774853625], 'margin': [0.845094647473231, 0.8525378546076124, 0.8500661000378826, 0.8701154924548856, 0.8809429008590184, 0.8919724789934765, 0.8974644035334682, 0.9007169814160916, 0.9087868066670086, 0.9127687100642763, 0.9216224169113791, 0.9225534589363809, 0.9270135437491126, 0.9335082931185272, 0.938530123024859, 0.93658925051

random - Iteration 1: 40.99% (3010/7344) samples labeled, f1_score: 0.8758 on Foot
random - Iteration 2: 41.98% (3083/7344) samples labeled, f1_score: 0.8770 on Foot
random - Iteration 3: 42.97% (3156/7344) samples labeled, f1_score: 0.8775 on Foot
random - Iteration 4: 43.97% (3229/7344) samples labeled, f1_score: 0.8780 on Foot
random - Iteration 5: 44.96% (3302/7344) samples labeled, f1_score: 0.8769 on Foot
random - Iteration 6: 45.96% (3375/7344) samples labeled, f1_score: 0.8835 on Foot
random - Iteration 7: 46.95% (3448/7344) samples labeled, f1_score: 0.8894 on Foot
random - Iteration 8: 47.94% (3521/7344) samples labeled, f1_score: 0.8866 on Foot
random - Iteration 9: 48.94% (3594/7344) samples labeled, f1_score: 0.8842 on Foot
random - Iteration 10: 49.93% (3667/7344) samples labeled, f1_score: 0.8885 on Foot
random - Iteration 11: 50.93% (3740/7344) samples labeled, f1_score: 0.8935 on Foot
random - Iteration 12: 51.92% (3813/7344) samples labeled, f1_score: 0.8919 on Foot
r

{'Foot': {'random': [0.8692775006075788, 0.8758456230339714, 0.8769692732761459, 0.877549359216981, 0.8780370205114096, 0.8769332678645776, 0.8834522645403133, 0.8893541873924189, 0.8865510755904626, 0.884249946365396, 0.8885394471379895, 0.8934620673367617, 0.8918613993109512, 0.8952663886502956, 0.8952926832178838, 0.8945960102465358, 0.9016308499161649, 0.9019587868748309, 0.8975605327661729, 0.904404131844147, 0.9069560851948804, 0.9085553994338132], 'margin': [0.8692775006075788, 0.8732434795970434, 0.8812016595778905, 0.8950561342143721, 0.892121136355278, 0.9020098584750272, 0.9108148221302477, 0.9218086426659843, 0.9227597991298543, 0.9217608320087016, 0.9295460852490726, 0.9361629050434935, 0.9372372904250538, 0.9403557612017422, 0.9353067980993396, 0.9393070953821594, 0.9436458083945801, 0.9469402554211716, 0.9469746527869081, 0.9454263599568989, 0.9480305374035503, 0.9453018053681361]}}
len(y_values_per_iteration)=22 pour random
len(y_values_per_iteration) après ajustement=2

 __évolution PRS avec différents % d'initialisation et batch ratio 1% mais BIAISED en donnant que des original flag = 1 en initialisation__

In [702]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"PRS": (X_PRS, y_PRS)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets_biased(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 0.20% (35/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (17765/17800)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (178/17800)


x_values pour labeled_ratio=0.002 : [0.2, 1.2, 2.1999999999999997, 3.2, 4.2, 5.2, 6.2, 7.200000000000001, 8.200000000000001, 9.2, 10.200000000000001, 11.200000000000001, 12.2, 13.200000000000001, 14.200000000000001, 15.2, 16.2, 17.200000000000003, 18.2, 19.2, 20.200000000000003, 21.2, 22.2, 23.200000000000003, 24.2, 25.2, 26.200000000000003, 27.200000000000003, 28.200000000000003, 29.2, 30.2, 31.2, 32.2, 33.2, 34.2, 35.2, 36.199999999999996, 37.2, 38.2, 39.2, 40.2, 41.2, 42.199999999999996, 43.2, 44.2, 45.2, 46.2, 47.2, 48.199999999999996, 49.2, 50.2, 51.2, 52.2, 53.2, 54.2, 55.2, 56.2, 57.2, 58.199999999999996, 59.199999999999996, 60.199999999999996]
X_biased[106](array([1.]), array([1629]))
y_biased(array([0., 1.]), array([ 526, 1103]))
shape de y_biased (1629,)
nb_labeled35
(array([0., 1.]), array([ 9, 26]))
Classes présentes dans xtrain 106: (array([1.]), array([35]))
Classes présentes dans y_train: (array([0., 1.]), array([ 9, 26]))
Shape de predict_proba: (4450, 2)


random - Iteration 1: 1.20% (213/17800) samples labeled, PR AUC: 0.4648 on PRS
random - Iteration 2: 2.20% (391/17800) samples labeled, PR AUC: 0.5877 on PRS
random - Iteration 3: 3.20% (569/17800) samples labeled, PR AUC: 0.6170 on PRS
random - Iteration 4: 4.20% (747/17800) samples labeled, PR AUC: 0.6398 on PRS
random - Iteration 5: 5.20% (925/17800) samples labeled, PR AUC: 0.6458 on PRS
random - Iteration 6: 6.20% (1103/17800) samples labeled, PR AUC: 0.6880 on PRS
random - Iteration 7: 7.20% (1281/17800) samples labeled, PR AUC: 0.6840 on PRS
random - Iteration 8: 8.20% (1459/17800) samples labeled, PR AUC: 0.6907 on PRS
random - Iteration 9: 9.20% (1637/17800) samples labeled, PR AUC: 0.6852 on PRS
random - Iteration 10: 10.20% (1815/17800) samples labeled, PR AUC: 0.7099 on PRS
random - Iteration 11: 11.20% (1993/17800) samples labeled, PR AUC: 0.7040 on PRS
random - Iteration 12: 12.20% (2171/17800) samples labeled, PR AUC: 0.7065 on PRS
random - Iteration 13: 13.20% (2349/178

{'PRS': {'random': [np.float64(0.4641847226897046), np.float64(0.46478656768711957), np.float64(0.5877269359517941), np.float64(0.6169795179097368), np.float64(0.6397581920304357), np.float64(0.6458258018987949), np.float64(0.6879870305440325), np.float64(0.6840167467787494), np.float64(0.6907358071137017), np.float64(0.685236921038924), np.float64(0.7099105413154918), np.float64(0.703979606590796), np.float64(0.7064691406574516), np.float64(0.7151557914136739), np.float64(0.7201109279840847), np.float64(0.724604298658953), np.float64(0.7110288315724153), np.float64(0.723780330710428), np.float64(0.7275367687488076), np.float64(0.7368637099556492), np.float64(0.7194739838762038), np.float64(0.7240468208519186), np.float64(0.7201102844953743), np.float64(0.7235429967395453), np.float64(0.7271665636103923), np.float64(0.7313118296068806), np.float64(0.7300169506181002), np.float64(0.7303145809930548), np.float64(0.7336297283303983), np.float64(0.739350075230299), np.float64(0.73623846070

random - Iteration 1: 2.00% (356/17800) samples labeled, PR AUC: 0.4718 on PRS
random - Iteration 2: 3.00% (534/17800) samples labeled, PR AUC: 0.6249 on PRS
random - Iteration 3: 4.00% (712/17800) samples labeled, PR AUC: 0.6614 on PRS
random - Iteration 4: 5.00% (890/17800) samples labeled, PR AUC: 0.6682 on PRS
random - Iteration 5: 6.00% (1068/17800) samples labeled, PR AUC: 0.6795 on PRS
random - Iteration 6: 7.00% (1246/17800) samples labeled, PR AUC: 0.6810 on PRS
random - Iteration 7: 8.00% (1424/17800) samples labeled, PR AUC: 0.7076 on PRS
random - Iteration 8: 9.00% (1602/17800) samples labeled, PR AUC: 0.7027 on PRS
random - Iteration 9: 10.00% (1780/17800) samples labeled, PR AUC: 0.7058 on PRS
random - Iteration 10: 11.00% (1958/17800) samples labeled, PR AUC: 0.7090 on PRS
random - Iteration 11: 12.00% (2136/17800) samples labeled, PR AUC: 0.7142 on PRS
random - Iteration 12: 13.00% (2314/17800) samples labeled, PR AUC: 0.7200 on PRS
random - Iteration 13: 14.00% (2492/1

...

{'PRS': {'random': [np.float64(0.39623129668383983), np.float64(0.38291441886513083), np.float64(0.660336476925343), np.float64(0.7059810542244809), np.float64(0.7040870278026166), np.float64(0.7171551563716647), np.float64(0.7315820138888269), np.float64(0.7342975006194706), np.float64(0.7424809606672168), np.float64(0.7446598514859593), np.float64(0.746724543742955), np.float64(0.7453461375049796), np.float64(0.747666794633717), np.float64(0.7546258883437862), np.float64(0.7553662469547742), np.float64(0.7537599081374787), np.float64(0.7525762406561027), np.float64(0.7440877337796239), np.float64(0.7514171151407723), np.float64(0.7503273825484553), np.float64(0.7495987620428695), np.float64(0.7477466658275107), np.float64(0.7553807120159844), np.float64(0.7501185047514896), np.float64(0.7477364056226526), np.float64(0.7549555146872864), np.float64(0.7583028304568828), np.float64(0.7569552957190315), np.float64(0.7566253070517592), np.float64(0.757517782364687), np.float64(0.763757674

random - Iteration 1: 10.15% (1807/17800) samples labeled, PR AUC: 0.3877 on PRS
random - Iteration 2: 11.15% (1985/17800) samples labeled, PR AUC: 0.6723 on PRS
random - Iteration 3: 12.15% (2163/17800) samples labeled, PR AUC: 0.6877 on PRS
random - Iteration 4: 13.15% (2341/17800) samples labeled, PR AUC: 0.6991 on PRS
random - Iteration 5: 14.15% (2519/17800) samples labeled, PR AUC: 0.7103 on PRS
random - Iteration 6: 15.15% (2697/17800) samples labeled, PR AUC: 0.7117 on PRS
random - Iteration 7: 16.15% (2875/17800) samples labeled, PR AUC: 0.7306 on PRS
random - Iteration 8: 17.15% (3053/17800) samples labeled, PR AUC: 0.7317 on PRS
random - Iteration 9: 18.15% (3231/17800) samples labeled, PR AUC: 0.7347 on PRS
random - Iteration 10: 19.15% (3409/17800) samples labeled, PR AUC: 0.7295 on PRS
random - Iteration 11: 20.15% (3587/17800) samples labeled, PR AUC: 0.7200 on PRS
random - Iteration 12: 21.15% (3765/17800) samples labeled, PR AUC: 0.7359 on PRS
random - Iteration 13: 22

{'PRS': {'random': [np.float64(0.37158808093650864), np.float64(0.3877137187036993), np.float64(0.6722540770989343), np.float64(0.6877364420603634), np.float64(0.6990931203581366), np.float64(0.7103385907591918), np.float64(0.7117128600374153), np.float64(0.7305718023521394), np.float64(0.7316937789728242), np.float64(0.7346565180679839), np.float64(0.7295328473448944), np.float64(0.7199641489645418), np.float64(0.7358841290855638), np.float64(0.738749859310656), np.float64(0.7374383681429237), np.float64(0.7331446502594343), np.float64(0.7417463599998977), np.float64(0.7442909151859947), np.float64(0.7376383613901564), np.float64(0.7467568371804536), np.float64(0.7392153551286422), np.float64(0.7445703904797323), np.float64(0.7407943533062162), np.float64(0.7376850093211267), np.float64(0.7411223470605919), np.float64(0.7492688285132587), np.float64(0.751465872599144), np.float64(0.7513104008404773), np.float64(0.7515734211622804), np.float64(0.7497877079044898), np.float64(0.75193972

random - Iteration 1: 10.15% (1807/17800) samples labeled, PR AUC: 0.3808 on PRS
random - Iteration 2: 11.15% (1985/17800) samples labeled, PR AUC: 0.6792 on PRS
random - Iteration 3: 12.15% (2163/17800) samples labeled, PR AUC: 0.6967 on PRS
random - Iteration 4: 13.15% (2341/17800) samples labeled, PR AUC: 0.7099 on PRS
random - Iteration 5: 14.15% (2519/17800) samples labeled, PR AUC: 0.7084 on PRS
random - Iteration 6: 15.15% (2697/17800) samples labeled, PR AUC: 0.7097 on PRS
random - Iteration 7: 16.15% (2875/17800) samples labeled, PR AUC: 0.7293 on PRS
random - Iteration 8: 17.15% (3053/17800) samples labeled, PR AUC: 0.7240 on PRS
random - Iteration 9: 18.15% (3231/17800) samples labeled, PR AUC: 0.7257 on PRS
random - Iteration 10: 19.15% (3409/17800) samples labeled, PR AUC: 0.7237 on PRS
random - Iteration 11: 20.15% (3587/17800) samples labeled, PR AUC: 0.7360 on PRS
random - Iteration 12: 21.15% (3765/17800) samples labeled, PR AUC: 0.7315 on PRS
random - Iteration 13: 22

{'PRS': {'random': [np.float64(0.39882447079623556), np.float64(0.3808233710152972), np.float64(0.679160026089463), np.float64(0.6967279857845229), np.float64(0.7098560503224298), np.float64(0.7084139465877353), np.float64(0.7097053613390587), np.float64(0.7292888614817863), np.float64(0.7240082056037696), np.float64(0.7257380350541637), np.float64(0.7236503397544521), np.float64(0.735988060209116), np.float64(0.731482398906125), np.float64(0.7298347738680397), np.float64(0.7269490397148825), np.float64(0.7417748836093522), np.float64(0.7314762070014094), np.float64(0.7357681188091345), np.float64(0.7385111102447339), np.float64(0.739691141463156), np.float64(0.7372254807136973), np.float64(0.7433991569934862)], 'margin': [np.float64(0.39882447079623556), np.float64(0.3983576344167716), np.float64(0.6049526186686399), np.float64(0.6775042563471991), np.float64(0.7290673964525629), np.float64(0.727257254425935), np.float64(0.7364946089503237), np.float64(0.7324470030675488), np.float64(

 __évolution PRS avec différents % d'initialisation et batch ratio 0.5% mais BIAISED en donnant que des original flag = 1 en initialisation__

In [704]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.005  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"PRS": (X_PRS, y_PRS)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets_biased(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 0.20% (35/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (17765/17800)
Nb d'itérations: 121
Nb de données labellisées en plus à chaque itération: 0.50% (89/17800)


x_values pour labeled_ratio=0.002 : [0.2, 0.7000000000000001, 1.2, 1.7000000000000002, 2.1999999999999997, 2.7, 3.2, 3.7000000000000006, 4.2, 4.7, 5.2, 5.7, 6.2, 6.7, 7.200000000000001, 7.7, 8.200000000000001, 8.700000000000001, 9.2, 9.700000000000001, 10.200000000000001, 10.7, 11.200000000000001, 11.700000000000001, 12.2, 12.7, 13.200000000000001, 13.700000000000001, 14.200000000000001, 14.7, 15.2, 15.7, 16.2, 16.7, 17.200000000000003, 17.700000000000003, 18.2, 18.7, 19.2, 19.7, 20.200000000000003, 20.700000000000003, 21.2, 21.7, 22.2, 22.7, 23.200000000000003, 23.700000000000003, 24.2, 24.7, 25.2, 25.7, 26.200000000000003, 26.700000000000003, 27.200000000000003, 27.700000000000003, 28.200000000000003, 28.700000000000003, 29.2, 29.7, 30.2, 30.7, 31.2, 31.7, 32.2, 32.7, 33.2, 33.7, 34.2, 34.7, 35.2, 35.699999999999996, 36.199999999999996, 36.7, 37.2, 37.7, 38.2, 38.7, 39.2, 39.7, 40.2, 40.7, 41.2, 41.7, 42.199999999999996, 42.699999999999996, 43.2, 43.7, 44.2, 44.7, 45.2, 45.7, 46.2, 4

random - Iteration 1: 0.70% (124/17800) samples labeled, PR AUC: 0.3680 on PRS
random - Iteration 2: 1.20% (213/17800) samples labeled, PR AUC: 0.6050 on PRS
random - Iteration 3: 1.70% (302/17800) samples labeled, PR AUC: 0.6233 on PRS
random - Iteration 4: 2.20% (391/17800) samples labeled, PR AUC: 0.6207 on PRS
random - Iteration 5: 2.70% (480/17800) samples labeled, PR AUC: 0.6464 on PRS
random - Iteration 6: 3.20% (569/17800) samples labeled, PR AUC: 0.6550 on PRS
random - Iteration 7: 3.70% (658/17800) samples labeled, PR AUC: 0.6558 on PRS
random - Iteration 8: 4.20% (747/17800) samples labeled, PR AUC: 0.6672 on PRS
random - Iteration 9: 4.70% (836/17800) samples labeled, PR AUC: 0.6620 on PRS
random - Iteration 10: 5.20% (925/17800) samples labeled, PR AUC: 0.6853 on PRS
random - Iteration 11: 5.70% (1014/17800) samples labeled, PR AUC: 0.6876 on PRS
random - Iteration 12: 6.20% (1103/17800) samples labeled, PR AUC: 0.6930 on PRS
random - Iteration 13: 6.70% (1192/17800) sampl

{'PRS': {'random': [np.float64(0.42019492987722695), np.float64(0.3680345174378973), np.float64(0.6050365122257735), np.float64(0.6233037559802191), np.float64(0.6207157636673774), np.float64(0.6463722235682989), np.float64(0.6550044996117307), np.float64(0.6557578261611664), np.float64(0.6672477496437542), np.float64(0.6619832490243007), np.float64(0.6853111397928224), np.float64(0.6875522275161152), np.float64(0.692995894541564), np.float64(0.7048046111783978), np.float64(0.6976197223052791), np.float64(0.7053096484346953), np.float64(0.7131599798163292), np.float64(0.711718685808129), np.float64(0.7035996689709139), np.float64(0.7029473716524691), np.float64(0.72208520870409), np.float64(0.7220992939852152), np.float64(0.7265846146074046), np.float64(0.7256846005299434), np.float64(0.7208939395527303), np.float64(0.7208681070078521), np.float64(0.7187582900776155), np.float64(0.7175913680762207), np.float64(0.726788314135886), np.float64(0.7270630711090089), np.float64(0.72427372842

random - Iteration 1: 1.50% (267/17800) samples labeled, PR AUC: 0.4504 on PRS
random - Iteration 2: 2.00% (356/17800) samples labeled, PR AUC: 0.6592 on PRS
random - Iteration 3: 2.50% (445/17800) samples labeled, PR AUC: 0.6721 on PRS
random - Iteration 4: 3.00% (534/17800) samples labeled, PR AUC: 0.6725 on PRS
random - Iteration 5: 3.50% (623/17800) samples labeled, PR AUC: 0.6718 on PRS
random - Iteration 6: 4.00% (712/17800) samples labeled, PR AUC: 0.6993 on PRS
random - Iteration 7: 4.50% (801/17800) samples labeled, PR AUC: 0.6910 on PRS
random - Iteration 8: 5.00% (890/17800) samples labeled, PR AUC: 0.6968 on PRS
random - Iteration 9: 5.50% (979/17800) samples labeled, PR AUC: 0.7099 on PRS
random - Iteration 10: 6.00% (1068/17800) samples labeled, PR AUC: 0.7080 on PRS
random - Iteration 11: 6.50% (1157/17800) samples labeled, PR AUC: 0.7194 on PRS
random - Iteration 12: 7.00% (1246/17800) samples labeled, PR AUC: 0.7058 on PRS
random - Iteration 13: 7.50% (1335/17800) samp

...

{'PRS': {'random': [np.float64(0.35835260427447696), np.float64(0.3611345257552444), np.float64(0.6123327847069926), np.float64(0.6554002767049126), np.float64(0.6696746137245551), np.float64(0.6887144693527153), np.float64(0.6971866663357186), np.float64(0.7016943720454035), np.float64(0.7090470815585491), np.float64(0.7145819884796663), np.float64(0.7290184796178474), np.float64(0.7282210720617741), np.float64(0.72923395111381), np.float64(0.7279266786906148), np.float64(0.7305794340366387), np.float64(0.7177634325133574), np.float64(0.7341386721484888), np.float64(0.7382569995265957), np.float64(0.7282693112592167), np.float64(0.7390984819293323), np.float64(0.7355876880628295), np.float64(0.731356093178287), np.float64(0.742535269578964), np.float64(0.7381369977860863), np.float64(0.743421547592274), np.float64(0.7343037592641848), np.float64(0.7398934934701353), np.float64(0.7442009074453798), np.float64(0.7406740073513849), np.float64(0.7460172567962711), np.float64(0.74530996253

random - Iteration 1: 9.65% (1718/17800) samples labeled, PR AUC: 0.4027 on PRS
random - Iteration 2: 10.15% (1807/17800) samples labeled, PR AUC: 0.6484 on PRS
random - Iteration 3: 10.65% (1896/17800) samples labeled, PR AUC: 0.6684 on PRS
random - Iteration 4: 11.15% (1985/17800) samples labeled, PR AUC: 0.6951 on PRS
random - Iteration 5: 11.65% (2074/17800) samples labeled, PR AUC: 0.7111 on PRS
random - Iteration 6: 12.15% (2163/17800) samples labeled, PR AUC: 0.7108 on PRS
random - Iteration 7: 12.65% (2252/17800) samples labeled, PR AUC: 0.7352 on PRS
random - Iteration 8: 13.15% (2341/17800) samples labeled, PR AUC: 0.7315 on PRS
random - Iteration 9: 13.65% (2430/17800) samples labeled, PR AUC: 0.7272 on PRS
random - Iteration 10: 14.15% (2519/17800) samples labeled, PR AUC: 0.7319 on PRS
random - Iteration 11: 14.65% (2608/17800) samples labeled, PR AUC: 0.7361 on PRS
random - Iteration 12: 15.15% (2697/17800) samples labeled, PR AUC: 0.7391 on PRS
random - Iteration 13: 15.

{'PRS': {'random': [np.float64(0.37979435380486515), np.float64(0.40266888113006705), np.float64(0.6484020250845737), np.float64(0.6684457233740271), np.float64(0.695070437131948), np.float64(0.7111049299703112), np.float64(0.7108342465937857), np.float64(0.7352151223482539), np.float64(0.7314589822029905), np.float64(0.7272191351950775), np.float64(0.7319389674806911), np.float64(0.7360744266940326), np.float64(0.7391300692426936), np.float64(0.7393208077230087), np.float64(0.7493721876480254), np.float64(0.7473340859115981), np.float64(0.7371474352868795), np.float64(0.7484582909912882), np.float64(0.7399917247555424), np.float64(0.7472198261243197), np.float64(0.7569962466373328), np.float64(0.7566100656894361), np.float64(0.743004027815258), np.float64(0.7457370965000907), np.float64(0.750552609472339), np.float64(0.7527818417846341), np.float64(0.7490951953197404), np.float64(0.750238750639663), np.float64(0.744004996819017), np.float64(0.7526334346958213), np.float64(0.7507103713

random - Iteration 1: 9.65% (1718/17800) samples labeled, PR AUC: 0.3962 on PRS
random - Iteration 2: 10.15% (1807/17800) samples labeled, PR AUC: 0.6409 on PRS
random - Iteration 3: 10.65% (1896/17800) samples labeled, PR AUC: 0.6602 on PRS
random - Iteration 4: 11.15% (1985/17800) samples labeled, PR AUC: 0.6780 on PRS
random - Iteration 5: 11.65% (2074/17800) samples labeled, PR AUC: 0.6979 on PRS
random - Iteration 6: 12.15% (2163/17800) samples labeled, PR AUC: 0.6962 on PRS
random - Iteration 7: 12.65% (2252/17800) samples labeled, PR AUC: 0.7073 on PRS
random - Iteration 8: 13.15% (2341/17800) samples labeled, PR AUC: 0.7159 on PRS
random - Iteration 9: 13.65% (2430/17800) samples labeled, PR AUC: 0.7262 on PRS
random - Iteration 10: 14.15% (2519/17800) samples labeled, PR AUC: 0.7300 on PRS
random - Iteration 11: 14.65% (2608/17800) samples labeled, PR AUC: 0.7391 on PRS
random - Iteration 12: 15.15% (2697/17800) samples labeled, PR AUC: 0.7286 on PRS
random - Iteration 13: 15.

{'PRS': {'random': [np.float64(0.3977913063791004), np.float64(0.396217692387867), np.float64(0.6408898450411886), np.float64(0.6602286188493756), np.float64(0.678002824396942), np.float64(0.6978709319207552), np.float64(0.6962203571436353), np.float64(0.7073004451432638), np.float64(0.7159426215865985), np.float64(0.726186559742378), np.float64(0.7300222216461023), np.float64(0.7390780627097481), np.float64(0.7285801984938124), np.float64(0.7301056171115303), np.float64(0.7376512322491182), np.float64(0.7395758131407096), np.float64(0.7399574947284432), np.float64(0.7430665034246413), np.float64(0.7490313406161491), np.float64(0.7435718352432215), np.float64(0.7479951323826256), np.float64(0.741344373554712), np.float64(0.7467179325169617), np.float64(0.7427049900981645), np.float64(0.7415262776041833), np.float64(0.747240270215465), np.float64(0.750987964301514), np.float64(0.7519212496613008), np.float64(0.7620703784037809), np.float64(0.7521241615539385), np.float64(0.7605771475395

 __évolution PRS avec différents % d'initialisation et batch ratio 1% mais BIAISED en donnant que des original flag = 0 en initialisation__

In [683]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"PRS": (X_PRS, y_PRS)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets_biased(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: PRS

Métrique utilisée: PR AUC


Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 0.20% (35/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (17765/17800)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (178/17800)


x_values pour labeled_ratio=0.002 : [0.2, 1.2, 2.1999999999999997, 3.2, 4.2, 5.2, 6.2, 7.200000000000001, 8.200000000000001, 9.2, 10.200000000000001, 11.200000000000001, 12.2, 13.200000000000001, 14.200000000000001, 15.2, 16.2, 17.200000000000003, 18.2, 19.2, 20.200000000000003, 21.2, 22.2, 23.200000000000003, 24.2, 25.2, 26.200000000000003, 27.200000000000003, 28.200000000000003, 29.2, 30.2, 31.2, 32.2, 33.2, 34.2, 35.2, 36.199999999999996, 37.2, 38.2, 39.2, 40.2, 41.2, 42.199999999999996, 43.2, 44.2, 45.2, 46.2, 47.2, 48.199999999999996, 49.2, 50.2, 51.2, 52.2, 53.2, 54.2, 55.2, 56.2, 57.2, 58.199999999999996, 59.199999999999996, 60.199999999999996]
X_biased[106](array([0.]), array([16171]))
y_biased(array([0., 1.]), array([14172,  1999]))
shape de y_biased (16171,)
nb_labeled35
(array([0., 1.]), array([32,  3]))
Classes présentes dans xtrain 106: (array([0.]), array([35]))
Classes présentes dans y_train: (array([0., 1.]), array([32,  3]))
Shape de predict_proba: (4450, 2)


random - Iteration 1: 1.20% (213/17800) samples labeled, PR AUC: 0.2197 on PRS
random - Iteration 2: 2.20% (391/17800) samples labeled, PR AUC: 0.6425 on PRS
random - Iteration 3: 3.20% (569/17800) samples labeled, PR AUC: 0.6619 on PRS
random - Iteration 4: 4.20% (747/17800) samples labeled, PR AUC: 0.6765 on PRS
random - Iteration 5: 5.20% (925/17800) samples labeled, PR AUC: 0.6830 on PRS
random - Iteration 6: 6.20% (1103/17800) samples labeled, PR AUC: 0.6682 on PRS
random - Iteration 7: 7.20% (1281/17800) samples labeled, PR AUC: 0.6778 on PRS
random - Iteration 8: 8.20% (1459/17800) samples labeled, PR AUC: 0.6794 on PRS
random - Iteration 9: 9.20% (1637/17800) samples labeled, PR AUC: 0.6841 on PRS
random - Iteration 10: 10.20% (1815/17800) samples labeled, PR AUC: 0.6887 on PRS
random - Iteration 11: 11.20% (1993/17800) samples labeled, PR AUC: 0.6794 on PRS
random - Iteration 12: 12.20% (2171/17800) samples labeled, PR AUC: 0.6994 on PRS
random - Iteration 13: 13.20% (2349/178

{'PRS': {'random': [np.float64(0.21261717111053127), np.float64(0.2196736319801736), np.float64(0.6424616239340615), np.float64(0.6618755327026908), np.float64(0.6765482094790086), np.float64(0.6829860174098078), np.float64(0.6681786078912042), np.float64(0.6777781552939859), np.float64(0.679355922199253), np.float64(0.6841286725585561), np.float64(0.6886913863457305), np.float64(0.6794483820803691), np.float64(0.6994260046200991), np.float64(0.703180777574449), np.float64(0.7181783898380542), np.float64(0.7241120432080936), np.float64(0.7230997143775858), np.float64(0.731554932012255), np.float64(0.7187962070306435), np.float64(0.7402522931205275), np.float64(0.7485445476679777), np.float64(0.7355847450975433), np.float64(0.7406693177259753), np.float64(0.7405713321528541), np.float64(0.7539237779588539), np.float64(0.7421737615002768), np.float64(0.7417883218496809), np.float64(0.747068404159), np.float64(0.7522600459195631), np.float64(0.7500926683058711), np.float64(0.7599722577507

...

{'PRS': {'random': [np.float64(0.6530793504931323), np.float64(0.6585494373687872), np.float64(0.6872608212625837), np.float64(0.6951752427728705), np.float64(0.7064893213127083), np.float64(0.7095571829317618), np.float64(0.7169487911042862), np.float64(0.7159944277001432), np.float64(0.7209184033642861), np.float64(0.7310811996172639), np.float64(0.7305758879066329), np.float64(0.7353694309431119), np.float64(0.7331999404072278), np.float64(0.7308614397414783), np.float64(0.7419084617220784), np.float64(0.7377327286949427), np.float64(0.7389133215206964), np.float64(0.750077189963559), np.float64(0.7498849770346109), np.float64(0.7490681145664464), np.float64(0.7463142880701508), np.float64(0.7460841271668458), np.float64(0.7514461982700216), np.float64(0.7461823518612125), np.float64(0.7505866751132846), np.float64(0.7522383726135465), np.float64(0.7501308405812436), np.float64(0.7514170007676758), np.float64(0.7562831086274026), np.float64(0.7515988740930215), np.float64(0.76087511

random - Iteration 1: 31.00% (5518/17800) samples labeled, PR AUC: 0.6631 on PRS
random - Iteration 2: 32.00% (5696/17800) samples labeled, PR AUC: 0.6898 on PRS
random - Iteration 3: 33.00% (5874/17800) samples labeled, PR AUC: 0.7132 on PRS
random - Iteration 4: 34.00% (6052/17800) samples labeled, PR AUC: 0.7018 on PRS
random - Iteration 5: 35.00% (6230/17800) samples labeled, PR AUC: 0.7217 on PRS
random - Iteration 6: 36.00% (6408/17800) samples labeled, PR AUC: 0.7340 on PRS
random - Iteration 7: 37.00% (6586/17800) samples labeled, PR AUC: 0.7390 on PRS
random - Iteration 8: 38.00% (6764/17800) samples labeled, PR AUC: 0.7368 on PRS
random - Iteration 9: 39.00% (6942/17800) samples labeled, PR AUC: 0.7439 on PRS
random - Iteration 10: 40.00% (7120/17800) samples labeled, PR AUC: 0.7344 on PRS
random - Iteration 11: 41.00% (7298/17800) samples labeled, PR AUC: 0.7472 on PRS
random - Iteration 12: 42.00% (7476/17800) samples labeled, PR AUC: 0.7433 on PRS
random - Iteration 13: 43

{'PRS': {'random': [np.float64(0.6574657458246919), np.float64(0.6630688661421759), np.float64(0.689792429364128), np.float64(0.7131546909851619), np.float64(0.7018315061312718), np.float64(0.7217225837188626), np.float64(0.7340000028467994), np.float64(0.7389769913578036), np.float64(0.736814023057655), np.float64(0.7439189903127967), np.float64(0.7343893447752309), np.float64(0.7472371719348064), np.float64(0.7433164703256019), np.float64(0.7537508114158913), np.float64(0.7518996676766759), np.float64(0.7481788769614554), np.float64(0.7447079596531481), np.float64(0.7556456311733695), np.float64(0.752687379427286), np.float64(0.7572241800075822), np.float64(0.7530570279764468), np.float64(0.7588891132855529), np.float64(0.7507694211869683), np.float64(0.749694346879213), np.float64(0.7537430403074691), np.float64(0.7510427518601102), np.float64(0.7599498451782918), np.float64(0.7523670629175462), np.float64(0.7564121819519485), np.float64(0.7644972214194984), np.float64(0.75959712369

random - Iteration 1: 41.00% (7298/17800) samples labeled, PR AUC: 0.6606 on PRS
random - Iteration 2: 42.00% (7476/17800) samples labeled, PR AUC: 0.6783 on PRS
random - Iteration 3: 43.00% (7654/17800) samples labeled, PR AUC: 0.7052 on PRS
random - Iteration 4: 44.00% (7832/17800) samples labeled, PR AUC: 0.7061 on PRS
random - Iteration 5: 45.00% (8010/17800) samples labeled, PR AUC: 0.7094 on PRS
random - Iteration 6: 46.00% (8188/17800) samples labeled, PR AUC: 0.7224 on PRS
random - Iteration 7: 47.00% (8366/17800) samples labeled, PR AUC: 0.7366 on PRS
random - Iteration 8: 48.00% (8544/17800) samples labeled, PR AUC: 0.7311 on PRS
random - Iteration 9: 49.00% (8722/17800) samples labeled, PR AUC: 0.7318 on PRS
random - Iteration 10: 50.00% (8900/17800) samples labeled, PR AUC: 0.7402 on PRS
random - Iteration 11: 51.00% (9078/17800) samples labeled, PR AUC: 0.7395 on PRS
random - Iteration 12: 52.00% (9256/17800) samples labeled, PR AUC: 0.7468 on PRS
random - Iteration 13: 53

{'PRS': {'random': [np.float64(0.6626756812790401), np.float64(0.6605687663176145), np.float64(0.6783375849176909), np.float64(0.7052341414272183), np.float64(0.7061480407258933), np.float64(0.7093823207880848), np.float64(0.7223689151828919), np.float64(0.7366375688057417), np.float64(0.7310646409543465), np.float64(0.7317744364356362), np.float64(0.7402000376537907), np.float64(0.7395494139209211), np.float64(0.7467826736336083), np.float64(0.739653037428105), np.float64(0.7397541159031339), np.float64(0.7418068919081601), np.float64(0.7299220381410298), np.float64(0.7387744830639174), np.float64(0.7527256222202209), np.float64(0.7426736952785062), np.float64(0.7476590506208747), np.float64(0.7519916106675509)], 'margin': [np.float64(0.6626756812790401), np.float64(0.664061789097022), np.float64(0.7066526608325062), np.float64(0.7344706961067732), np.float64(0.7468643017856794), np.float64(0.7566131458950219), np.float64(0.7552695160984756), np.float64(0.7518856065480515), np.float64

 __évolution PRS avec différents % d'initialisation et batch ratio 0.5% mais BIAISED en donnant que des original flag = 0 en initialisation__

In [724]:
# Liste des ratios de données labellisées à tester
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.005  # Taille du batch d'échantillons ajoutés à chaque itération

# Dictionnaire pour stocker le nombre d'itérations max pour chaque labeled_ratio
iterations_list = {}

# Calculer le nombre d'itérations maximum pour chaque labeled_ratio
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    iterations_list[labeled_ratio] = max_iterations

methods = ["random","margin"]
model_class = lambda: RandomForestClassifier()

# Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),  
    clone(LogisticRegression(max_iter=1000)), 
    clone(SVC(probability=True)),      
]

datasets = {"PRS": (X_PRS, y_PRS)}  # Exemple de dataset
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

# Créer un graphique pour l'évolution de l'accuracy
for dataset_name in datasets:
    metric = METRICS[dataset_name]
    
    # Créer un graphique vierge pour chaque dataset
    fig = px.line(title=f"Évolution de l'{metric.__name__ if hasattr(metric, '__name__') else str(metric)} sur {dataset_name}")

    y_values = []
    
    # Boucle pour chaque labeled_ratio
    for labeled_ratio in labeled_ratios:
        max_iterations = iterations_list[labeled_ratio]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]
        print(f"x_values pour labeled_ratio={labeled_ratio} : {x_values}")

        # Exécuter l'expérience d'Active Learning pour obtenir les résultats
        results = run_active_learning_experiment_datasets_biased(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations, batch_ratio, methods, model_class, models
        )
        print(results)

        for method in methods:
            # Récupérer les valeurs de y (accuracy pour chaque itération et méthode)
            y_values_per_iteration = results[dataset_name][method]
            print(f"len(y_values_per_iteration)={len(y_values_per_iteration)} pour {method}")

            # Vérification que x et y ont la même taille
            if len(y_values_per_iteration) < len(x_values):
                y_values_per_iteration += [None] * (len(x_values) - len(y_values_per_iteration))
            
            print(f"len(y_values_per_iteration) après ajustement={len(y_values_per_iteration)}")

            y_values.append(y_values_per_iteration)

            # Ajouter les résultats au graphique
            fig.add_scatter(
                x=x_values, y=y_values_per_iteration, mode="lines+markers", 
                name=f"{method} pour {dataset_name} (labeled ratio {labeled_ratio})", connectgaps=True
            )

    # Déterminer les limites de l'axe Y en aplatissant la liste des valeurs de y
    y_flat = [val for sublist in y_values for val in sublist if val is not None]
    min_y = min(y_flat) if y_flat else 0
    max_y = max(y_flat) if y_flat else 1

    # Mise en forme du graphique : axes et légende
    fig.update_layout(
        xaxis_title="% du training set labellisé",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 60]),  # Plage de l'axe X de 0 à 60%
        yaxis=dict(range=[min_y, max_y])  # Plage de l'axe Y ajustée dynamiquement
    )

    # Afficher le graphique
    fig.show()



Traitement du dataset: PRS

Métrique utilisée: PR AUC
Taille totale du dataset PRS: 100.00% (22250/22250)
Taille de l'ensemble de test: 20.00% (4450/22250)
Taille de l'ensemble de training: 80.00% (17800/22250)
Taille de l'ensemble labellisé dans le training set: 0.20% (35/17800)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (17765/17800)
Nb d'itérations: 121
Nb de données labellisées en plus à chaque itération: 0.50% (89/17800)


x_values pour labeled_ratio=0.002 : [0.2, 0.7000000000000001, 1.2, 1.7000000000000002, 2.1999999999999997, 2.7, 3.2, 3.7000000000000006, 4.2, 4.7, 5.2, 5.7, 6.2, 6.7, 7.200000000000001, 7.7, 8.200000000000001, 8.700000000000001, 9.2, 9.700000000000001, 10.200000000000001, 10.7, 11.200000000000001, 11.700000000000001, 12.2, 12.7, 13.200000000000001, 13.700000000000001, 14.200000000000001, 14.7, 15.2, 15.7, 16.2, 16.7, 17.200000000000003, 17.700000000000003, 18.2, 18.7, 19.2, 19.7, 20.200000000000003, 20.700000000000003, 21.2, 21.7, 22.2, 22.7, 23.200000000000003, 23.700000000000003, 24.2, 24.7, 25.2, 25.7, 26.200000000000003, 26.700000000000003, 27.200000000000003, 27.700000000000003, 28.200000000000003, 28.700000000000003, 29.2, 29.7, 30.2, 30.7, 31.2, 31.7, 32.2, 32.7, 33.2, 33.7, 34.2, 34.7, 35.2, 35.699999999999996, 36.199999999999996, 36.7, 37.2, 37.7, 38.2, 38.7, 39.2, 39.7, 40.2, 40.7, 41.2, 41.7, 42.199999999999996, 42.699999999999996, 43.2, 43.7, 44.2, 44.7, 45.2, 45.7, 46.2, 4

random - Iteration 1: 0.70% (124/17800) samples labeled, PR AUC: 0.2119 on PRS
random - Iteration 2: 1.20% (213/17800) samples labeled, PR AUC: 0.5006 on PRS
random - Iteration 3: 1.70% (302/17800) samples labeled, PR AUC: 0.5817 on PRS
random - Iteration 4: 2.20% (391/17800) samples labeled, PR AUC: 0.6315 on PRS
random - Iteration 5: 2.70% (480/17800) samples labeled, PR AUC: 0.6373 on PRS
random - Iteration 6: 3.20% (569/17800) samples labeled, PR AUC: 0.6436 on PRS
random - Iteration 7: 3.70% (658/17800) samples labeled, PR AUC: 0.6668 on PRS
random - Iteration 8: 4.20% (747/17800) samples labeled, PR AUC: 0.6712 on PRS
random - Iteration 9: 4.70% (836/17800) samples labeled, PR AUC: 0.6567 on PRS
random - Iteration 10: 5.20% (925/17800) samples labeled, PR AUC: 0.6631 on PRS
random - Iteration 11: 5.70% (1014/17800) samples labeled, PR AUC: 0.6607 on PRS
random - Iteration 12: 6.20% (1103/17800) samples labeled, PR AUC: 0.6642 on PRS
random - Iteration 13: 6.70% (1192/17800) sampl

{'PRS': {'random': [np.float64(0.22018680606894703), np.float64(0.2119278216306885), np.float64(0.5006372000346705), np.float64(0.5816667525518495), np.float64(0.6314857010970821), np.float64(0.6373165783108274), np.float64(0.6435921900200835), np.float64(0.6667565610995018), np.float64(0.6711624239326501), np.float64(0.6567303305652448), np.float64(0.66305200954588), np.float64(0.6606561516489137), np.float64(0.6642446241750486), np.float64(0.6500457774860289), np.float64(0.6581018665978486), np.float64(0.6735523033482902), np.float64(0.6798533024996105), np.float64(0.6785604498657644), np.float64(0.6738066812494803), np.float64(0.6799620886224709), np.float64(0.6828736289261419), np.float64(0.698264879438206), np.float64(0.6985328246402418), np.float64(0.6981676878484929), np.float64(0.7048742932787911), np.float64(0.7030079715494926), np.float64(0.7056819594571792), np.float64(0.7095747791274643), np.float64(0.7103565829884627), np.float64(0.7079778937682748), np.float64(0.714090962

random - Iteration 1: 1.50% (267/17800) samples labeled, PR AUC: 0.5295 on PRS
random - Iteration 2: 2.00% (356/17800) samples labeled, PR AUC: 0.5661 on PRS
random - Iteration 3: 2.50% (445/17800) samples labeled, PR AUC: 0.6158 on PRS
random - Iteration 4: 3.00% (534/17800) samples labeled, PR AUC: 0.6356 on PRS
random - Iteration 5: 3.50% (623/17800) samples labeled, PR AUC: 0.6528 on PRS
random - Iteration 6: 4.00% (712/17800) samples labeled, PR AUC: 0.6472 on PRS
random - Iteration 7: 4.50% (801/17800) samples labeled, PR AUC: 0.6567 on PRS
random - Iteration 8: 5.00% (890/17800) samples labeled, PR AUC: 0.6557 on PRS
random - Iteration 9: 5.50% (979/17800) samples labeled, PR AUC: 0.6503 on PRS
random - Iteration 10: 6.00% (1068/17800) samples labeled, PR AUC: 0.6657 on PRS
random - Iteration 11: 6.50% (1157/17800) samples labeled, PR AUC: 0.6547 on PRS
random - Iteration 12: 7.00% (1246/17800) samples labeled, PR AUC: 0.6629 on PRS
random - Iteration 13: 7.50% (1335/17800) samp

...

{'PRS': {'random': [np.float64(0.6575629338219076), np.float64(0.6558522868365003), np.float64(0.6697686752809068), np.float64(0.6666154564015809), np.float64(0.6800426923833285), np.float64(0.680174520578326), np.float64(0.6960974342237924), np.float64(0.6993441941872458), np.float64(0.6991612299895109), np.float64(0.70402072206369), np.float64(0.7159341666023049), np.float64(0.702543050540003), np.float64(0.7171701705871776), np.float64(0.7063979547956649), np.float64(0.7024446862196252), np.float64(0.7153530896823808), np.float64(0.7310389215584521), np.float64(0.7226576211486484), np.float64(0.712720634258133), np.float64(0.7158097474397569), np.float64(0.7180582301887618), np.float64(0.7188573163031176), np.float64(0.7244824115875252), np.float64(0.7282579966381635), np.float64(0.7286084690631994), np.float64(0.7411629233167982), np.float64(0.7366462649946263), np.float64(0.7228514430392103), np.float64(0.7325757639809629), np.float64(0.7332774232247083), np.float64(0.727162040512

random - Iteration 1: 30.50% (5429/17800) samples labeled, PR AUC: 0.6603 on PRS
random - Iteration 2: 31.00% (5518/17800) samples labeled, PR AUC: 0.6695 on PRS
random - Iteration 3: 31.50% (5607/17800) samples labeled, PR AUC: 0.6748 on PRS
random - Iteration 4: 32.00% (5696/17800) samples labeled, PR AUC: 0.6874 on PRS
random - Iteration 5: 32.50% (5785/17800) samples labeled, PR AUC: 0.6922 on PRS
random - Iteration 6: 33.00% (5874/17800) samples labeled, PR AUC: 0.6976 on PRS
random - Iteration 7: 33.50% (5963/17800) samples labeled, PR AUC: 0.7080 on PRS
random - Iteration 8: 34.00% (6052/17800) samples labeled, PR AUC: 0.7104 on PRS
random - Iteration 9: 34.50% (6141/17800) samples labeled, PR AUC: 0.7083 on PRS
random - Iteration 10: 35.00% (6230/17800) samples labeled, PR AUC: 0.7174 on PRS
random - Iteration 11: 35.50% (6319/17800) samples labeled, PR AUC: 0.7183 on PRS
random - Iteration 12: 36.00% (6408/17800) samples labeled, PR AUC: 0.7253 on PRS
random - Iteration 13: 36

{'PRS': {'random': [np.float64(0.6537603879063998), np.float64(0.6602533727019051), np.float64(0.6694711892955553), np.float64(0.6747924437828796), np.float64(0.6874047287057753), np.float64(0.6921810333106331), np.float64(0.6976111104742875), np.float64(0.7080230290489561), np.float64(0.710405390187081), np.float64(0.70833103433993), np.float64(0.7173612162570876), np.float64(0.718264908449032), np.float64(0.7253124120496479), np.float64(0.7269622242171754), np.float64(0.7266412720440966), np.float64(0.7234098828715915), np.float64(0.7332140284688516), np.float64(0.7321949744667948), np.float64(0.7335219574180015), np.float64(0.7317609666495393), np.float64(0.7366740511489157), np.float64(0.7384151588105519), np.float64(0.7373639797715578), np.float64(0.7272385409578896), np.float64(0.741433137514105), np.float64(0.7505390025253237), np.float64(0.7351641528636192), np.float64(0.7466921000751878), np.float64(0.7386975088543024), np.float64(0.7396914774754816), np.float64(0.745394300104

random - Iteration 1: 40.50% (7209/17800) samples labeled, PR AUC: 0.6651 on PRS
random - Iteration 2: 41.00% (7298/17800) samples labeled, PR AUC: 0.6808 on PRS
random - Iteration 3: 41.50% (7387/17800) samples labeled, PR AUC: 0.6901 on PRS
random - Iteration 4: 42.00% (7476/17800) samples labeled, PR AUC: 0.7034 on PRS
random - Iteration 5: 42.50% (7565/17800) samples labeled, PR AUC: 0.6853 on PRS
random - Iteration 6: 43.00% (7654/17800) samples labeled, PR AUC: 0.6976 on PRS
random - Iteration 7: 43.50% (7743/17800) samples labeled, PR AUC: 0.7094 on PRS
random - Iteration 8: 44.00% (7832/17800) samples labeled, PR AUC: 0.7042 on PRS
random - Iteration 9: 44.50% (7921/17800) samples labeled, PR AUC: 0.7061 on PRS
random - Iteration 10: 45.00% (8010/17800) samples labeled, PR AUC: 0.7192 on PRS
random - Iteration 11: 45.50% (8099/17800) samples labeled, PR AUC: 0.7197 on PRS
random - Iteration 12: 46.00% (8188/17800) samples labeled, PR AUC: 0.7138 on PRS
random - Iteration 13: 46

{'PRS': {'random': [np.float64(0.6606138530037604), np.float64(0.6650659059930208), np.float64(0.6807912529888457), np.float64(0.6901025902516806), np.float64(0.7034169302589486), np.float64(0.6852543577763494), np.float64(0.6976278668155275), np.float64(0.7093691862709884), np.float64(0.7041931810130592), np.float64(0.7061359367363635), np.float64(0.7191970942740643), np.float64(0.7196770060551692), np.float64(0.7138499713528592), np.float64(0.7229931382974599), np.float64(0.7238725454428461), np.float64(0.7224836134181786), np.float64(0.7287546263845983), np.float64(0.72366000055485), np.float64(0.7261116898221519), np.float64(0.7301280071095099), np.float64(0.7320757072256896), np.float64(0.7310581021279546), np.float64(0.7344680304877531), np.float64(0.7346205223069898), np.float64(0.7409156720589763), np.float64(0.7411345061841191), np.float64(0.7411354500881465), np.float64(0.7372684328338857), np.float64(0.7455249042611868), np.float64(0.7382166002549789), np.float64(0.742879573

Brouillon

In [735]:
import time
import plotly.express as px
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import clone

# Paramètres globaux
labeled_ratios = [0.002,0.01,0.02 ,0.05, 0.1,0.2,0.3,0.4]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

methods = ["random"]  
model_class = lambda: RandomForestClassifier() 

#Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),
    clone(LogisticRegression(max_iter=1000)),
    clone(SVC(probability=True)),
]

#  Datasets et métriques
datasets = {"MNIST": (X_MNIST, y_MNIST)}  # Exemple de dataset
METRICS = {"MNIST": f1_score, "Foot": f1_score, "PRS": "PR AUC"}  # Métriques

#  Stockage des temps d'exécution
execution_times = {}

#  Boucle sur les batch_ratios
for labeled_ratio in labeled_ratios:
    max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    execution_times[labeled_ratio] = {}

    for dataset_name in datasets:
        metric = METRICS[dataset_name]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]

        # ⏳ Chronométrage
        start_time = time.time()

        # Lancer l'expérience Active Learning
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations,
            batch_ratio, methods, model_class, models
        )

        elapsed_time = time.time() - start_time  # Temps d'exécution
        execution_times[labeled_ratio] = elapsed_time

# 📊 Création du DataFrame des temps d'exécution
df_times = pd.DataFrame.from_dict(execution_times, orient="index")  # ✅ Structure correcte
df_times.index.name = "Labeled Ratio"

# 📌 Affichage du tableau des temps
print(df_times)

# 🎨 📈 **Graphique des temps d'exécution**
fig_time = px.line(
    df_times, x=df_times.index, y=df_times.columns, markers=True,
    title="⏳ Temps d'exécution en fonction de Labeled_ratio",
    labels={"index": "Labeled Ratio", "value": "Temps (s)"},
)

fig_time.update_traces(line=dict(width=3))  # Épaissir les lignes
fig_time.update_layout(
    xaxis=dict(title="Labeled Ratio", tickmode="linear", dtick=0.01),
    yaxis_title="Temps (s)",
    legend_title="Dataset",
    template="plotly_dark",  # Thème sympa
)

fig_time.show()



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 0.20% (16/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.80% (7984/8000)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
random - Iteration 1: 1.20% (96/8000) samples labeled, f1_score: 0.3844 on MNIST
random - Iteration 2: 2.20% (176/8000) samples labeled, f1_score: 0.7345 on MNIST
random - Iteration 3: 3.20% (256/8000) samples labeled, f1_score: 0.7821 on MNIST
random - Iteration 4: 4.20% (336/8000) samples labeled, f1_score: 0.8186 on MNIST
random - Iteration 5: 5.20% (416/8000) samples labeled, f1_score: 0.8428 on MNIST
random - Iteration 6: 6.20% (496/8000) samples labeled, f1_score: 0.8552 on MNIST
random - Iteration 7: 7.20% (576/8000) samples labeled, f

                       0
Labeled Ratio           
0.002          35.117882
0.010          34.162241
0.020          33.943772
0.050          34.780142
0.100          34.903685
0.200          31.493558
0.300          28.838315
0.400          21.621633


In [36]:
import time
import plotly.express as px
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import clone

# Paramètres globaux
labeled_ratios =  [0.005,0.01,0.02 ,0.05, 0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5]
test_ratio = 0.2  # Proportion du dataset réservée au test
batch_ratio = 0.01  # Taille du batch d'échantillons ajoutés à chaque itération

methods = ["random","margin"]  
model_class = lambda: RandomForestClassifier() 

#Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),
    clone(LogisticRegression(max_iter=1000)),
    clone(SVC(probability=True)),
]

#  Datasets et métriques
datasets = {"MNIST": (X_MNIST, y_MNIST)}  # Exemple de dataset
METRICS = {"MNIST": f1_score, "Foot": f1_score, "PRS": "PR AUC"}  # Métriques


execution_times = {method: {} for method in methods}

# 🔄 Boucle sur les méthodes
for method in methods:
    method_list = [method]

    #  Boucle sur les batch_ratios
    for labeled_ratio in labeled_ratios:
        max_iterations = int((0.60 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%

        for dataset_name in datasets:
            metric = METRICS[dataset_name]

            # Générer les valeurs de x (pourcentage de données labellisées)
            x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]

            # ⏳ Chronométrage
            start_time = time.time()

            # Lancer l'expérience Active Learning
            results = run_active_learning_experiment_datasets(
                datasets, METRICS, labeled_ratio, test_ratio, max_iterations,
                batch_ratio, method_list, model_class, models
            )

            elapsed_time = time.time() - start_time  # Temps d'exécution
            execution_times[method][labeled_ratio] = elapsed_time

# 📊 Conversion en DataFrame multi-index
df_times = pd.DataFrame(execution_times)
df_times.index.name = "Label Ratio"
df_times.reset_index(inplace=True)  # Pour avoir une structure exploitable dans plotly

# 📌 Affichage du tableau des temps
print(df_times)

# 🎨 📈 **Graphique des temps d'exécution par méthode**
fig_time = px.line(
    df_times, x="Label Ratio", y=methods, markers=True,
    title="⏳ Temps d'exécution en fonction du label_ratio",
    labels={"Label Ratio": "Label Ratio", "value": "Temps (s)", "variable": "Méthode"},
)

fig_time.update_traces(line=dict(width=3))  # Épaissir les lignes
fig_time.update_layout(
    xaxis=dict(tickmode="linear", dtick=0.1),
    yaxis_title="Temps d'exécution (s)",
    legend_title="Méthode",
)

fig_time.show()




Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 0.50% (40/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.50% (7960/8000)
Nb d'itérations: 61
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (random) on MNIST: 0.9429



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 1.00% (80/8000)
Taille de l'ensemble non-labellisé dans le training set: 99.00% (7920/8000)
Nb d'itérations: 60
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (random) on MNIST: 0.9460



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 2.00% (160/8000)
Taille de l'ensemble non-labellisé dans le training set: 98.00% (7840/8000)
Nb d'itérations: 59
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (random) on MNIST: 0.9394



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 5.00% (400/8000)
Taille de l'ensemble non-labellisé dans le training set: 95.00% (7600/8000)
Nb d'itérations: 56
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (random) on MNIST: 0.9480



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 52
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (random) on MNIST: 0.9464


...


Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 35.00% (2800/8000)
Taille de l'ensemble non-labellisé dans le training set: 65.00% (5200/8000)
Nb d'itérations: 27
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (margin) on MNIST: 0.9592



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 40.00% (3200/8000)
Taille de l'ensemble non-labellisé dans le training set: 60.00% (4800/8000)
Nb d'itérations: 21
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (margin) on MNIST: 0.9547



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 45.00% (3600/8000)
Taille de l'ensemble non-labellisé dans le training set: 55.00% (4400/8000)
Nb d'itérations: 16
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (margin) on MNIST: 0.9547



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 50.00% (4000/8000)
Taille de l'ensemble non-labellisé dans le training set: 50.00% (4000/8000)
Nb d'itérations: 11
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)
Final f1_score (margin) on MNIST: 0.9536


    Label Ratio     random     margin
0         0.005  36.335867  42.022699
1         0.010  36.157416  42.158172
2         0.020  35.377978  42.125957
3         0.050  34.606882  40.988392
4         0.100  34.807357  40.257685
5         0.150  32.985751  37.204544
6         0.200  31.138588  35.889074
7         0.250  30.403104  34.740131
8         0.300  28.166259  30.844911
9         0.350  25.608041  27.769360
10        0.400  21.329821  23.350033
11        0.450  17.986949  18.927905
12        0.500  13.856907  14.453652


In [37]:
import time
import plotly.express as px
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import clone

# 📌 Paramètres globaux
labeled_ratio = 0.1  
test_ratio = 0.2 
batch_ratios = [0.005,0.01,0.02 ,0.05, 0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5]
methods = ["random", "margin"]  
model_class = lambda: RandomForestClassifier()

# 📌 Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),
    clone(LogisticRegression(max_iter=1000)),
    clone(SVC(probability=True)),
]

# 📌 Datasets et métriques
datasets = {"MNIST": (X_MNIST, y_MNIST)}  # Exemple de dataset
METRICS = {"MNIST": f1_score, "Foot": f1_score, "PRS": "PR AUC"}  

# 📌 Stockage des temps d'exécution par méthode et batch_ratio
execution_times = {method: {} for method in methods}

# 🔄 Boucle sur les méthodes
for method in methods:
    method_list = [method]

    # 🔄 Boucle sur les batch_ratios
    for batch_ratio in batch_ratios:
        max_iterations = int((1 - labeled_ratio) / batch_ratio) + 2  

        # ⏳ Chronométrage
        start_time = time.time()

        # Lancer l'expérience Active Learning
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations,
            batch_ratio, method_list, model_class, models
        )

        elapsed_time = time.time() - start_time  # Temps d'exécution

        # ✅ Stocker le temps d'exécution pour cette méthode et ce batch_ratio
        execution_times[method][batch_ratio] = elapsed_time

# 📊 Conversion en DataFrame multi-index
df_times = pd.DataFrame(execution_times)
df_times.index.name = "Batch Ratio"
df_times.reset_index(inplace=True)  # Pour avoir une structure exploitable dans plotly

# 📌 Affichage du tableau des temps
print(df_times)

# 🎨 📈 **Graphique des temps d'exécution par méthode**
fig_time = px.line(
    df_times, x="Batch Ratio", y=methods, markers=True,
    title="⏳ Temps d'exécution en fonction du batch_ratio",
    labels={"Batch Ratio": "Batch Ratio", "value": "Temps (s)", "variable": "Méthode"},
)

fig_time.update_traces(line=dict(width=3))  # Épaissir les lignes
fig_time.update_layout(
    xaxis=dict(tickmode="linear", dtick=0.1),
    yaxis_title="Temps d'exécution (s)",
    legend_title="Méthode",  # Thème visuel
)

fig_time.show()



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 182
Nb de données labellisées en plus à chaque itération: 0.50% (40/8000)

Toutes les données ont été labellisées après 180 itérations.
Final f1_score (random) on MNIST: 0.9502



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 92
Nb de données labellisées en plus à chaque itération: 1.00% (80/8000)

Toutes les données ont été labellisées après 90 itérations.
Final f1_score (random) on MNIST: 0.9526



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 47
Nb de données labellisées en plus à chaque itération: 2.00% (160/8000)

Toutes les données ont été labellisées après 45 itérations.
Final f1_score (random) on MNIST: 0.9519



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 20
Nb de données labellisées en plus à chaque itération: 5.00% (400/8000)

Toutes les données ont été labellisées après 18 itérations.
Final f1_score (random) on MNIST: 0.9500



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 11
Nb de données labellisées en plus à chaque itération: 10.00% (800/8000)

Toutes les données ont été labellisées après 9 itérations.
Final f1_score (random) on MNIST: 0.9549


...


Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 4
Nb de données labellisées en plus à chaque itération: 35.00% (2800/8000)

Batch réduit à 1600 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 3 itérations.
Final f1_score (margin) on MNIST: 0.9536



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 4
Nb de données labellisées en plus à chaque itération: 40.00% (3200/8000)

Batch réduit à 800 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 3 itérations.
Final f1_score (margin) on MNIST: 0.9562



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 4
Nb de données labellisées en plus à chaque itération: 45.00% (3600/8000)

Toutes les données ont été labellisées après 2 itérations.
Final f1_score (margin) on MNIST: 0.9520



Traitement du dataset: MNIST

Métrique utilisée: f1_score
Taille totale du dataset MNIST: 100.00% (10000/10000)
Taille de l'ensemble de test: 20.00% (2000/10000)
Taille de l'ensemble de training: 80.00% (8000/10000)
Taille de l'ensemble labellisé dans le training set: 10.00% (800/8000)
Taille de l'ensemble non-labellisé dans le training set: 90.00% (7200/8000)
Nb d'itérations: 3
Nb de données labellisées en plus à chaque itération: 50.00% (4000/8000)

Batch réduit à 3200 échantillons car la pool est presque vide.

Toutes les données ont été labellisées après 2 itérations.
Final f1_score (margin) on MNIST: 0.9535


    Batch Ratio      random      margin
0         0.005  189.902248  204.319493
1         0.010   97.647751  104.184583
2         0.020   50.477469   53.554132
3         0.050   21.489710   23.274873
4         0.100   12.614967   13.160671
5         0.150    9.323918   10.008346
6         0.200    8.832446    9.304855
7         0.250    7.723826    8.080105
8         0.300    6.409031    6.758162
9         0.350    6.721852    7.236714
10        0.400    7.024041    7.183228
11        0.450    5.379600    5.584564
12        0.500    5.485794    5.694440


In [ ]:
import time
import plotly.express as px
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import clone

# Paramètres globaux
labeled_ratio = 0.1  
test_ratio = 0.2 
batch_ratios = [0.005, 0.01, 0.02, 0.05]  
methods = ["random"]  
model_class = lambda: RandomForestClassifier() 

#Liste des modèles pour le comité (si nécessaire)
models = [
    clone(RandomForestClassifier()),
    clone(LogisticRegression(max_iter=1000)),
    clone(SVC(probability=True)),
]

#  Datasets et métriques
datasets = {"MNIST": (X_MNIST, y_MNIST)}  # Exemple de dataset
METRICS = {"MNIST": f1_score, "Foot": f1_score, "PRS": "PR AUC"}  # Métriques

#  Stockage des temps d'exécution
execution_times = {}

#  Boucle sur les batch_ratios
for batch_ratio in batch_ratios:
    max_iterations = int((0.30 - labeled_ratio) / batch_ratio) + 2  # Ne pas dépasser 60%
    execution_times[batch_ratio] = {}
    
    for dataset_name in datasets:
        metric = METRICS[dataset_name]

        # Générer les valeurs de x (pourcentage de données labellisées)
        x_values = [(labeled_ratio + i * batch_ratio) * 100 for i in range(max_iterations)]

        # ⏳ Chronométrage
        start_time = time.time()

        # Lancer l'expérience Active Learning
        results = run_active_learning_experiment_datasets(
            datasets, METRICS, labeled_ratio, test_ratio, max_iterations,
            batch_ratio, methods, model_class, models
        )

        elapsed_time = time.time() - start_time  # Temps d'exécution
        execution_times[batch_ratio] = elapsed_time

# 📊 Création du DataFrame des temps d'exécution
df_times = pd.DataFrame.from_dict(execution_times, orient="index")  # ✅ Structure correcte
df_times.index.name = "Batch Ratio"

# 📌 Affichage du tableau des temps
print(df_times)

# 🎨 📈 **Graphique des temps d'exécution**
fig_time = px.line(
    df_times, x=df_times.index, y=df_times.columns, markers=True,
    title="⏳ Temps d'exécution en fonction de batch_ratio",
    labels={"index": "Batch Ratio", "value": "Temps (s)"},
)

fig_time.update_traces(line=dict(width=3))  # Épaissir les lignes
fig_time.update_layout(
    xaxis=dict(title="Batch Ratio", tickmode="linear", dtick=0.01),
    yaxis_title="Temps (s)",
    legend_title="Dataset",
    template="plotly_dark",  # Thème sympa
)

fig_time.show()
